<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

In [1]:

import copy
import math
import torch
import numpy as np
from scipy import stats as scipy_stats
from transformers import GPT2Tokenizer, GPT2LMHeadModel

MODEL_NAME = 'gpt2'
TARGET_LAYER = 0          # which MLP block (0-indexed) to permute
DEVICE = 'cpu'            # 'cuda' if available

# Fixed IID evaluation corpus for steps 1 and 4 (swap in a larger held-out set for a
# real run -- this is intentionally small so the notebook is fast to read through).
EVAL_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a subfield of artificial intelligence.",
    "Machine learning models learn patterns from large amounts of data.",
    "The transformer architecture relies on self-attention mechanisms.",
    "Scientists discovered a new species of frog in the rainforest.",
    "The stock market experienced significant volatility this week.",
    "Cooking pasta requires boiling water and a pinch of salt.",
    "The history of Rome spans over a thousand years.",
]

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {MODEL_NAME}, target layer: {TARGET_LAYER}, device: {DEVICE}")
print(f"Eval corpus: {len(EVAL_CORPUS)} sentences")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: gpt2, target layer: 0, device: cpu
Eval corpus: 8 sentences


# baseline

In [2]:
def next_token_accuracy(model, tokenizer, texts, device='cpu'):
    accs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits
            preds = logits[0, :-1, :].argmax(dim=-1)
            targets = ids[0, 1:]
            accs.append((preds == targets).float().mean().item())
    return float(np.mean(accs))


def perplexity(model, tokenizer, texts, device='cpu'):
    losses = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))


def expected_calibration_error(model, tokenizer, texts, device='cpu', n_bins=10):
    confidences, corrects = [], []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits[0, :-1, :]
            targets = ids[0, 1:]
            probs = torch.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)
            confidences.extend(conf.tolist())
            corrects.extend((pred == targets).tolist())

    confidences = np.array(confidences)
    corrects = np.array(corrects).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = corrects[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return float(ece)


def run_battery(model, tokenizer, texts, device='cpu', label='model'):
    acc = next_token_accuracy(model, tokenizer, texts, device)
    ppl = perplexity(model, tokenizer, texts, device)
    ece = expected_calibration_error(model, tokenizer, texts, device)
    print(f"{label:>12s} | accuracy={acc:.4f}  perplexity={ppl:.3f}  ECE={ece:.4f}")
    return {'label': label, 'accuracy': acc, 'perplexity': ppl, 'ece': ece}


baseline_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_model.eval()

print("=" * 70)
print("STEP 1: BASELINE")
print("=" * 70)
baseline_result = run_battery(baseline_model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

STEP 1: BASELINE


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


    baseline | accuracy=0.3155  perplexity=56.629  ECE=0.1050


# aligned permutation

In [3]:
def get_mlp_layer(model, layer_idx):
    return model.transformer.h[layer_idx].mlp


def make_aligned_permutation(model, layer_idx, seed=0, device='cpu'):
    """Deep-copies `model` and permutes the d_mlp hidden axis of one MLP layer,
    applying the SAME permutation to c_fc's output and c_proj's input."""
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)

    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), perm] = 1.0

    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        mlp.c_proj.weight.data = P.T @ mlp.c_proj.weight.data  # compensate: P^-1 = P^T

    return m, perm


print("=" * 70)
print("STEP 2: ALIGNED PERMUTATION")
print("=" * 70)

aligned_model, perm = make_aligned_permutation(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)

# sanity check: logits should match to numerical precision on a single example
with torch.no_grad():
    sample_ids = tokenizer(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    base_logits = baseline_model(sample_ids).logits
    aligned_logits = aligned_model(sample_ids).logits
    max_abs_diff = (base_logits - aligned_logits).abs().max().item()

print(f"Layer {TARGET_LAYER} permutation applied ({perm.numel()} neurons).")
print(f"Max abs logit difference vs. baseline on sample sentence: {max_abs_diff:.2e}")
print("(should be ~1e-5 or smaller -- floating point noise, not a real difference)")

STEP 2: ALIGNED PERMUTATION


Layer 0 permutation applied (3072 neurons).
Max abs logit difference vs. baseline on sample sentence: 7.63e-05
(should be ~1e-5 or smaller -- floating point noise, not a real difference)


# unaligned permutation

In [4]:
def make_unaligned_permutation(model, layer_idx, seed=0, device='cpu'):
    """Same permutation as make_aligned_permutation, but WITHOUT compensating
    c_proj -- deliberately breaks the neuron correspondence."""
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)

    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), perm] = 1.0

    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        # c_proj intentionally left untouched -- this is the broken-symmetry control

    return m, perm


print("=" * 70)
print("STEP 3: UNALIGNED PERMUTATION (NEGATIVE CONTROL)")
print("=" * 70)

unaligned_model, _ = make_unaligned_permutation(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)

with torch.no_grad():
    sample_ids = tokenizer(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    base_logits = baseline_model(sample_ids).logits
    unaligned_logits = unaligned_model(sample_ids).logits
    max_abs_diff = (base_logits - unaligned_logits).abs().max().item()

print(f"Layer {TARGET_LAYER} permutation applied WITHOUT c_proj compensation.")
print(f"Max abs logit difference vs. baseline on sample sentence: {max_abs_diff:.3f}")
print("(should be large -- this is the intended collapse)")


STEP 3: UNALIGNED PERMUTATION (NEGATIVE CONTROL)


Layer 0 permutation applied WITHOUT c_proj compensation.
Max abs logit difference vs. baseline on sample sentence: 53.760
(should be large -- this is the intended collapse)


# aligned and unaligned vs baseline

In [5]:
print("=" * 70)
print("STEP 4: FULL BATTERY -- BASELINE vs. ALIGNED vs. UNALIGNED")
print("=" * 70)

aligned_result = run_battery(aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='aligned')
unaligned_result = run_battery(unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='unaligned')

print("\nSummary (diff from baseline):")
for r in [aligned_result, unaligned_result]:
    d_acc = r['accuracy'] - baseline_result['accuracy']
    d_ppl = r['perplexity'] - baseline_result['perplexity']
    d_ece = r['ece'] - baseline_result['ece']
    print(f"{r['label']:>10s} | d(accuracy)={d_acc:+.4f}  d(perplexity)={d_ppl:+.3f}  d(ECE)={d_ece:+.4f}")

print("\nExpected: aligned diffs ~0 across all three metrics; unaligned diffs large.")

STEP 4: FULL BATTERY -- BASELINE vs. ALIGNED vs. UNALIGNED


     aligned | accuracy=0.3155  perplexity=56.629  ECE=0.1050


   unaligned | accuracy=0.0409  perplexity=3892.809  ECE=0.1055

Summary (diff from baseline):
   aligned | d(accuracy)=+0.0000  d(perplexity)=-0.000  d(ECE)=-0.0000
 unaligned | d(accuracy)=-0.2746  d(perplexity)=+3836.180  d(ECE)=+0.0004

Expected: aligned diffs ~0 across all three metrics; unaligned diffs large.


# representations- linear CKA

In [6]:
def linear_cka(X, Y):
    """Linear CKA between two (n_samples, n_features) matrices."""
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    xty = X.T @ Y
    xtx = X.T @ X
    yty = Y.T @ Y
    num = (xty ** 2).sum()
    den = np.sqrt((xtx ** 2).sum() * (yty ** 2).sum())
    return float(num / den) if den > 0 else 0.0


def collect_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            rows.append(out.hidden_states[layer][0].cpu().numpy())
    return np.concatenate(rows, axis=0)


print("=" * 70)
print("STEP 5: LINEAR CKA -- BASELINE vs. ALIGNED / UNALIGNED, PER LAYER")
print("=" * 70)

num_layers = baseline_model.config.n_layer + 1  # +1 for embedding layer
for layer in range(num_layers):
    h_base = collect_hidden_states(baseline_model, tokenizer, EVAL_CORPUS, layer, DEVICE)
    h_aligned = collect_hidden_states(aligned_model, tokenizer, EVAL_CORPUS, layer, DEVICE)
    h_unaligned = collect_hidden_states(unaligned_model, tokenizer, EVAL_CORPUS, layer, DEVICE)

    cka_aligned = linear_cka(h_base, h_aligned)
    cka_unaligned = linear_cka(h_base, h_unaligned)
    print(f"Layer {layer:2d}: CKA(baseline, aligned)={cka_aligned:.4f}   "
          f"CKA(baseline, unaligned)={cka_unaligned:.4f}")

print("\nExpected: CKA(aligned) ~1.0 at every layer; CKA(unaligned) drops sharply from")
print("the manipulated layer onward.")


STEP 5: LINEAR CKA -- BASELINE vs. ALIGNED / UNALIGNED, PER LAYER


Layer  0: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=1.0000


Layer  1: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.7794


Layer  2: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9668


Layer  3: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9145


...

Layer  8: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9376


Layer  9: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9396


Layer 10: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9413


Layer 11: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.9445


Layer 12: CKA(baseline, aligned)=1.0000   CKA(baseline, unaligned)=0.0400

Expected: CKA(aligned) ~1.0 at every layer; CKA(unaligned) drops sharply from
the manipulated layer onward.


# activation patching

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
CAUSAL_TRACE_PAIRS = [
    ("The capital of France is", " Paris", "The capital of Germany is", " Berlin"),
    ("The opposite of hot is", " cold", "The opposite of up is", " down"),
    ("A dog says", " woof", "A cat says", " meow"),
]


def logit_diff(logits, correct_id, wrong_id):
    return (logits[0, -1, correct_id] - logits[0, -1, wrong_id]).item()


def run_causal_tracing(model_with_output_hidden_states, tokenizer, pairs, device='cpu'):
    model = model_with_output_hidden_states
    n_layers = model.config.n_layer
    restoration_by_layer = np.zeros(n_layers)
    used_pairs = 0

    for clean_prompt, correct_ans, corrupted_prompt, wrong_ans in pairs:
        correct_id = tokenizer.encode(correct_ans)[0]
        wrong_id = tokenizer.encode(wrong_ans)[0]

        clean_ids = tokenizer(clean_prompt, return_tensors='pt').input_ids.to(device)
        corrupt_ids = tokenizer(corrupted_prompt, return_tensors='pt').input_ids.to(device)
        if clean_ids.shape[1] != corrupt_ids.shape[1]:
            continue  # keep only length-matched pairs so token positions line up
        used_pairs += 1

        with torch.no_grad():
            clean_out = model(clean_ids, output_hidden_states=True)
            clean_cache = clean_out.hidden_states
            clean_logits = clean_out.logits
            corrupt_logits = model(corrupt_ids).logits

        clean_diff = logit_diff(clean_logits, correct_id, wrong_id)
        corrupt_diff = logit_diff(corrupt_logits, correct_id, wrong_id)
        subject_pos = clean_ids.shape[1] - 2  # token just before "is"

        for layer in range(n_layers):
            def hook(module, inp, out, layer=layer):
                out = list(out) if isinstance(out, tuple) else [out]
                out[0][:, subject_pos, :] = clean_cache[layer + 1][:, subject_pos, :]
                return tuple(out) if isinstance(out, tuple) else out[0]

            h = model.transformer.h[layer].register_forward_hook(hook)
            with torch.no_grad():
                patched_logits = model(corrupt_ids).logits
            h.remove()

            patched_diff = logit_diff(patched_logits, correct_id, wrong_id)
            denom = (clean_diff - corrupt_diff) if (clean_diff - corrupt_diff) != 0 else 1e-8
            restoration_by_layer[layer] += (patched_diff - corrupt_diff) / denom

    if used_pairs > 0:
        restoration_by_layer /= used_pairs
    return restoration_by_layer


print("=" * 70)
print("STEP 6: CAUSAL TRACING -- WHICH LAYER CARRIES THE FACT AFTER UNALIGNED PERMUTATION")
print("=" * 70)

# patch clean (baseline) activations into the UNALIGNED model's corrupted run --
# checks whether the layer that should carry the fact still can, once corrected
unaligned_for_tracing = GPT2LMHeadModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(DEVICE)
unaligned_for_tracing.load_state_dict(unaligned_model.state_dict())
unaligned_for_tracing.eval()

restoration = run_causal_tracing(unaligned_for_tracing, tokenizer, CAUSAL_TRACE_PAIRS, DEVICE)
for layer, val in enumerate(restoration):
    print(f"Layer {layer:2d}: mean restoration = {val:.3f}")

print("\nA restoration score near 1.0 at a given layer means patching that layer's clean")
print("activation back in recovers the correct answer -- i.e., that's where the fact")
print("lives in the unaligned (broken) model's forward pass.")


STEP 6: CAUSAL TRACING -- WHICH LAYER CARRIES THE FACT AFTER UNALIGNED PERMUTATION


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IndexError: Dimension out of range (expected to be in range of [-2, 1], but got 2)

# out of distribution

In [ ]:
OOD_CORPUS = {
    'code': [
        "def factorial(n):\n    if n == 0:\n        return 1\n    return n * factorial(n - 1)",
        "for i in range(10):\n    print(i ** 2)",
        "{\"name\": \"example\", \"values\": [1, 2, 3], \"active\": true}",
    ],
    'non_english': [
        "La vida es un viaje lleno de sorpresas inesperadas.",
        "Der schnelle braune Fuchs springt uber den faulen Hund.",
        "Le soleil se leve a l'est chaque matin sans exception.",
    ],
    'corrupted': [
        "the dog quickly the over jumps fox brown",
        "xk3f9 random tokens zzq without any coherent meaning qwerty",
        "was is the of a to in on at by for with",
    ],
}


def ood_perplexity_report(model, tokenizer, ood_corpus, device='cpu', label='model'):
    print(f"\n-- {label} --")
    results = {}
    for domain, texts in ood_corpus.items():
        ppl = perplexity(model, tokenizer, texts, device)
        results[domain] = ppl
        print(f"  {domain:>12s}: perplexity = {ppl:8.2f}")
    return results


print("=" * 70)
print("STEP 7: OOD STRESS TEST")
print("=" * 70)

ood_baseline = ood_perplexity_report(baseline_model, tokenizer, OOD_CORPUS, DEVICE, 'baseline')
ood_aligned = ood_perplexity_report(aligned_model, tokenizer, OOD_CORPUS, DEVICE, 'aligned')
ood_unaligned = ood_perplexity_report(unaligned_model, tokenizer, OOD_CORPUS, DEVICE, 'unaligned')

print("\nExpected: aligned perplexity tracks baseline closely on every OOD domain;")
print("unaligned perplexity spikes, showing the collapse isn't just an IID-prose effect.")


# multi-seeds

In [ ]:
N_SEEDS = 15
CONFIDENCE = 0.95


def seed_sweep(model, tokenizer, texts, layer_idx, n_seeds, device='cpu'):
    aligned_diffs, unaligned_diffs = [], []
    baseline_acc = next_token_accuracy(model, tokenizer, texts, device)

    for seed in range(n_seeds):
        aligned, _ = make_aligned_permutation(model, layer_idx, seed=seed, device=device)
        unaligned, _ = make_unaligned_permutation(model, layer_idx, seed=seed, device=device)
        aligned_diffs.append(next_token_accuracy(aligned, tokenizer, texts, device) - baseline_acc)
        unaligned_diffs.append(next_token_accuracy(unaligned, tokenizer, texts, device) - baseline_acc)

    return np.array(aligned_diffs), np.array(unaligned_diffs)


def mean_ci(values, confidence=0.95):
    mean = values.mean()
    sem = scipy_stats.sem(values) if len(values) > 1 else 0.0
    ci = scipy_stats.t.interval(confidence, len(values) - 1, loc=mean, scale=sem) if sem > 0 else (mean, mean)
    return float(mean), (float(ci[0]), float(ci[1]))


print("=" * 70)
print(f"STEP 8: MULTI-SEED SWEEP (N={N_SEEDS} seeds, {int(CONFIDENCE*100)}% CI)")
print("=" * 70)

aligned_diffs, unaligned_diffs = seed_sweep(baseline_model, tokenizer, EVAL_CORPUS, TARGET_LAYER, N_SEEDS, DEVICE)

aligned_mean, aligned_ci = mean_ci(aligned_diffs, CONFIDENCE)
unaligned_mean, unaligned_ci = mean_ci(unaligned_diffs, CONFIDENCE)

print(f"Aligned   accuracy diff: mean={aligned_mean:+.5f}  CI=[{aligned_ci[0]:+.5f}, {aligned_ci[1]:+.5f}]")
print(f"Unaligned accuracy diff: mean={unaligned_mean:+.5f}  CI=[{unaligned_ci[0]:+.5f}, {unaligned_ci[1]:+.5f}]")

print("\nExpected: aligned CI tightly bracketing 0; unaligned CI bracketing a large")
print("negative value, well separated from the aligned CI.")


In [ ]:

import copy
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.linalg import orthogonal_procrustes
from transformers import GPT2Tokenizer, GPT2LMHeadModel

MODEL_NAME = 'gpt2'
TARGET_LAYER = 0
DEVICE = 'cpu'
N_SEEDS = 15
CONFIDENCE = 0.95

EVAL_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a subfield of artificial intelligence.",
    "Machine learning models learn patterns from large amounts of data.",
    "The transformer architecture relies on self-attention mechanisms.",
    "Scientists discovered a new species of frog in the rainforest.",
    "The stock market experienced significant volatility this week.",
    "Cooking pasta requires boiling water and a pinch of salt.",
    "The history of Rome spans over a thousand years.",
]

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})
COLORS = {
    'baseline': '#2b2b2b',
    'aligned': '#1f77b4',
    'unaligned': '#d62728',
    'head_aligned': '#2ca02c',
    'head_unaligned': '#ff7f0e',
    'rotation_local': '#9467bd',
    'rotation_global': '#8c564b',
    'noise': '#e377c2',
    'deletion': '#17becf',
}

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

baseline_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_model.eval()


# --- core battery (from L0) ---
def next_token_accuracy(model, tokenizer, texts, device='cpu'):
    accs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits
            preds = logits[0, :-1, :].argmax(dim=-1)
            targets = ids[0, 1:]
            accs.append((preds == targets).float().mean().item())
    return float(np.mean(accs))


def perplexity(model, tokenizer, texts, device='cpu'):
    losses = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))


def expected_calibration_error(model, tokenizer, texts, device='cpu', n_bins=10):
    confidences, corrects = [], []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits[0, :-1, :]
            targets = ids[0, 1:]
            probs = torch.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)
            confidences.extend(conf.tolist())
            corrects.extend((pred == targets).tolist())
    confidences = np.array(confidences)
    corrects = np.array(corrects).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(corrects[mask].mean() - confidences[mask].mean())
    return float(ece)


def run_battery(model, tokenizer, texts, device='cpu', label='model', verbose=True):
    acc = next_token_accuracy(model, tokenizer, texts, device)
    ppl = perplexity(model, tokenizer, texts, device)
    ece = expected_calibration_error(model, tokenizer, texts, device)
    if verbose:
        print(f"{label:>18s} | accuracy={acc:.4f}  perplexity={ppl:8.3f}  ECE={ece:.4f}")
    return {'label': label, 'accuracy': acc, 'perplexity': ppl, 'ece': ece}


def get_mlp_layer(model, layer_idx):
    return model.transformer.h[layer_idx].mlp


def make_aligned_permutation(model, layer_idx, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), perm] = 1.0
    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        mlp.c_proj.weight.data = P.T @ mlp.c_proj.weight.data
    return m, perm


def make_unaligned_permutation(model, layer_idx, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), perm] = 1.0
    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
    return m, perm


def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    num = ((X.T @ Y) ** 2).sum()
    den = np.sqrt(((X.T @ X) ** 2).sum() * ((Y.T @ Y) ** 2).sum())
    return float(num / den) if den > 0 else 0.0


def collect_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            rows.append(out.hidden_states[layer][0].cpu().numpy())
    return np.concatenate(rows, axis=0)


print("Setup complete.")
baseline_result = run_battery(baseline_model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline (L0 rerun check)')


# attention head permutations

In [ ]:
def build_head_block_permutation(num_heads, head_dim, perm, device='cpu'):
    embed_dim = num_heads * head_dim
    P = torch.zeros(embed_dim, embed_dim, device=device)
    for new_pos, old_head in enumerate(perm):
        rows = torch.arange(old_head * head_dim, (old_head + 1) * head_dim)
        cols = torch.arange(new_pos * head_dim, (new_pos + 1) * head_dim)
        P[rows, cols] = 1.0
    return P


def permute_attention_heads(model, layer_idx=0, aligned=True, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    attn = m.transformer.h[layer_idx].attn
    num_heads, embed_dim = attn.num_heads, attn.embed_dim
    head_dim = embed_dim // num_heads

    torch.manual_seed(seed)
    perm = torch.randperm(num_heads).tolist()
    P = build_head_block_permutation(num_heads, head_dim, perm, device=device)

    with torch.no_grad():
        w, b = attn.c_attn.weight.data, attn.c_attn.bias.data
        segments = [0, 1, 2] if aligned else [2]  # Q,K,V if aligned; V only if unaligned
        for seg in segments:
            sl = slice(seg * embed_dim, (seg + 1) * embed_dim)
            w[:, sl] = w[:, sl] @ P
            b[sl] = b[sl] @ P
        if aligned:
            attn.c_proj.weight.data = P.T @ attn.c_proj.weight.data
    return m, perm


print("=" * 78)
print("SECTION 2: ATTENTION-HEAD PERMUTATION -- BASELINE vs. ALIGNED vs. UNALIGNED")
print("=" * 78)

head_aligned_model, head_perm = permute_attention_heads(baseline_model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
head_unaligned_model, _ = permute_attention_heads(baseline_model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)

with torch.no_grad():
    sample_ids = tokenizer(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    base_logits = baseline_model(sample_ids).logits
    ha_diff = (base_logits - head_aligned_model(sample_ids).logits).abs().max().item()
    hu_diff = (base_logits - head_unaligned_model(sample_ids).logits).abs().max().item()

print(f"Layer {TARGET_LAYER}, head permutation: {head_perm}")
print(f"Max abs logit diff, aligned:   {ha_diff:.2e}  (expect ~1e-5, floating-point noise)")
print(f"Max abs logit diff, unaligned: {hu_diff:.3f}  (expect large)\n")

head_baseline_result = run_battery(baseline_model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline')
head_aligned_result = run_battery(head_aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='head-aligned')
head_unaligned_result = run_battery(head_unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='head-unaligned')

# alternative perturbations

In [ ]:
def make_local_orthogonal_transform(model, layer_idx, seed=0, device='cpu'):
    """Generic orthogonal Q on the d_mlp axis, aligned the same way as permutation."""
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    d_mlp = mlp.c_fc.nf
    torch.manual_seed(seed)
    A = torch.randn(d_mlp, d_mlp, device=device)
    Q, _ = torch.linalg.qr(A)
    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ Q
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ Q
        mlp.c_proj.weight.data = Q.T @ mlp.c_proj.weight.data
    return m


def make_global_rotation(model, seed=0, device='cpu'):
    """Single orthogonal Q conjugating the whole residual stream; LayerNorm untouched."""
    m = copy.deepcopy(model)
    n_embd = m.config.n_embd
    torch.manual_seed(seed)
    A = torch.randn(n_embd, n_embd, device=device)
    Q, _ = torch.linalg.qr(A)
    with torch.no_grad():
        m.transformer.wte.weight.data = m.transformer.wte.weight.data @ Q
        m.transformer.wpe.weight.data = m.transformer.wpe.weight.data @ Q
        for block in m.transformer.h:
            block.attn.c_attn.weight.data = Q.T @ block.attn.c_attn.weight.data
            block.attn.c_proj.weight.data = block.attn.c_proj.weight.data @ Q
            block.mlp.c_fc.weight.data = Q.T @ block.mlp.c_fc.weight.data
            block.mlp.c_proj.weight.data = block.mlp.c_proj.weight.data @ Q
    return m


def make_gaussian_noise_model(model, layer_idx, sigma_frac=0.1, seed=0, device='cpu'):
    """Adds N(0, (sigma_frac * weight_std)^2) noise to c_fc and c_proj of one MLP layer."""
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    torch.manual_seed(seed)
    with torch.no_grad():
        for w in [mlp.c_fc.weight.data, mlp.c_proj.weight.data]:
            sigma = sigma_frac * w.std()
            w.add_(torch.randn_like(w) * sigma)
    return m


def make_neuron_deletion_model(model, layer_idx, delete_frac=0.1, seed=0, device='cpu'):
    """Zeroes out delete_frac of the d_mlp hidden units (rows of c_fc's output, i.e.
    columns of its weight; correspondingly zeroes the matching rows of c_proj)."""
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    d_mlp = mlp.c_fc.nf
    torch.manual_seed(seed)
    n_delete = int(round(d_mlp * delete_frac))
    delete_idx = torch.randperm(d_mlp)[:n_delete]
    with torch.no_grad():
        mlp.c_fc.weight.data[:, delete_idx] = 0.0
        mlp.c_fc.bias.data[delete_idx] = 0.0
        mlp.c_proj.weight.data[delete_idx, :] = 0.0
    return m


print("=" * 78)
print("SECTION 3: ALTERNATIVE PERTURBATION CONTROLS vs. ALIGNED PERMUTATION")
print("=" * 78)

aligned_model, _ = make_aligned_permutation(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)
local_rot_model = make_local_orthogonal_transform(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)
global_rot_model = make_global_rotation(baseline_model, seed=0, device=DEVICE)

noise_models = {f'gaussian_noise (sigma={s:.2f}x)': make_gaussian_noise_model(baseline_model, TARGET_LAYER, sigma_frac=s, seed=0, device=DEVICE)
                 for s in [0.05, 0.1, 0.25]}
deletion_models = {f'neuron_deletion ({int(f*100)}%)': make_neuron_deletion_model(baseline_model, TARGET_LAYER, delete_frac=f, seed=0, device=DEVICE)
                    for f in [0.1, 0.25, 0.5]}

perturbation_results = []
perturbation_results.append(run_battery(baseline_model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline'))
perturbation_results.append(run_battery(aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='aligned permutation'))
perturbation_results.append(run_battery(local_rot_model, tokenizer, EVAL_CORPUS, DEVICE, label='local rotation (MLP axis)'))
perturbation_results.append(run_battery(global_rot_model, tokenizer, EVAL_CORPUS, DEVICE, label='global rotation (residual)'))
for label, m in noise_models.items():
    perturbation_results.append(run_battery(m, tokenizer, EVAL_CORPUS, DEVICE, label=label))
for label, m in deletion_models.items():
    perturbation_results.append(run_battery(m, tokenizer, EVAL_CORPUS, DEVICE, label=label))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels = [r['label'] for r in perturbation_results]
accs = [r['accuracy'] for r in perturbation_results]
ppls = [r['perplexity'] for r in perturbation_results]
bar_colors = ['#2b2b2b', '#1f77b4'] + ['#9467bd', '#8c564b'] + ['#e377c2'] * len(noise_models) + ['#17becf'] * len(deletion_models)

axes[0].barh(labels, accs, color=bar_colors)
axes[0].axvline(baseline_result['accuracy'], color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('Next-token accuracy')
axes[0].set_title('Accuracy under each perturbation')
axes[0].invert_yaxis()

axes[1].barh(labels, np.log10(ppls), color=bar_colors)
axes[1].axvline(np.log10(baseline_result['perplexity']), color='gray', linestyle='--', linewidth=1)
axes[1].set_xlabel('log10(Perplexity)')
axes[1].set_title('Perplexity (log scale) under each perturbation')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('l1_section3_perturbation_controls.png', dpi=150, bbox_inches='tight')
plt.show()


# identity boundary

In [ ]:
def partial_violation_permutation(model, layer_idx, violation_frac, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    mlp = get_mlp_layer(m, layer_idx)
    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf

    perm = torch.randperm(d_mlp)               # applied to c_fc's output
    inverse_perm = torch.argsort(perm)          # correct compensation for c_proj's input

    comp_perm = inverse_perm.clone()
    n_violate = int(round(d_mlp * violation_frac))
    if n_violate >= 2:
        violate_positions = torch.randperm(d_mlp)[:n_violate]
        shuffled_values = comp_perm[violate_positions][torch.randperm(n_violate)]
        comp_perm[violate_positions] = shuffled_values

    with torch.no_grad():
        P = torch.zeros(d_mlp, d_mlp, device=device)
        P[torch.arange(d_mlp), perm] = 1.0
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        mlp.c_proj.weight.data = mlp.c_proj.weight.data[comp_perm, :]

    return m


print("=" * 78)
print("SECTION 4: IDENTITY BOUNDARY ANALYSIS -- GRADED SYMMETRY VIOLATION")
print("=" * 78)

VIOLATION_FRACS = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
N_BOUNDARY_SEEDS = 5  # average over a few seeds per violation level to smooth the curve

boundary_records = []
for v in VIOLATION_FRACS:
    accs, ppls = [], []
    for seed in range(N_BOUNDARY_SEEDS):
        m = partial_violation_permutation(baseline_model, TARGET_LAYER, v, seed=seed, device=DEVICE)
        accs.append(next_token_accuracy(m, tokenizer, EVAL_CORPUS, DEVICE))
        ppls.append(perplexity(m, tokenizer, EVAL_CORPUS, DEVICE))
    rec = {'violation_frac': v, 'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
           'ppl_mean': float(np.mean(ppls)), 'ppl_std': float(np.std(ppls))}
    boundary_records.append(rec)
    print(f"violation={v*100:5.0f}% | accuracy={rec['acc_mean']:.4f} +/- {rec['acc_std']:.4f}  "
          f"perplexity={rec['ppl_mean']:9.2f} +/- {rec['ppl_std']:7.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
v_pct = [r['violation_frac'] * 100 for r in boundary_records]
acc_means = [r['acc_mean'] for r in boundary_records]
acc_stds = [r['acc_std'] for r in boundary_records]
ppl_means = [r['ppl_mean'] for r in boundary_records]
ppl_stds = [r['ppl_std'] for r in boundary_records]

axes[0].errorbar(v_pct, acc_means, yerr=acc_stds, marker='o', color=COLORS['aligned'], capsize=3)
axes[0].axhline(baseline_result['accuracy'], color='gray', linestyle='--', linewidth=1, label='baseline')
axes[0].set_xlabel('Symmetry violation (%)')
axes[0].set_ylabel('Next-token accuracy')
axes[0].set_title('Identity degradation: accuracy')
axes[0].legend()

axes[1].errorbar(v_pct, ppl_means, yerr=ppl_stds, marker='o', color=COLORS['unaligned'], capsize=3)
axes[1].axhline(baseline_result['perplexity'], color='gray', linestyle='--', linewidth=1, label='baseline')
axes[1].set_yscale('log')
axes[1].set_xlabel('Symmetry violation (%)')
axes[1].set_ylabel('Perplexity (log scale)')
axes[1].set_title('Identity degradation: perplexity')
axes[1].legend()

plt.tight_layout()
plt.savefig('l1_section4_identity_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

# representation geom

In [ ]:
def procrustes_distance(X, Y):
    """Normalized residual after the best orthogonal alignment of X onto Y."""
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    X = X / (np.linalg.norm(X) + 1e-8)
    Y = Y / (np.linalg.norm(Y) + 1e-8)
    R, _ = orthogonal_procrustes(X, Y)
    residual = np.linalg.norm(X @ R - Y)
    return float(residual)


def top_k_principal_angles(hidden_a, hidden_b, k=10):
    a = hidden_a - hidden_a.mean(axis=0, keepdims=True)
    b = hidden_b - hidden_b.mean(axis=0, keepdims=True)
    _, _, Va = np.linalg.svd(a, full_matrices=False)
    _, _, Vb = np.linalg.svd(b, full_matrices=False)
    k = min(k, Va.shape[0], Vb.shape[0])
    Va_k, Vb_k = Va[:k].T, Vb[:k].T
    s = np.linalg.svd(Va_k.T @ Vb_k, compute_uv=False)
    s = np.clip(s, -1.0, 1.0)
    return np.degrees(np.arccos(s))


print("=" * 78)
print("SECTION 5: REPRESENTATION GEOMETRY -- CKA, PROCRUSTES, PRINCIPAL ANGLES")
print("=" * 78)

geometry_conditions = {
    'aligned': aligned_model,
    'unaligned': make_unaligned_permutation(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)[0],
    'head_aligned': head_aligned_model,
    'head_unaligned': head_unaligned_model,
    'global_rotation': global_rot_model,
}

num_layers = baseline_model.config.n_layer + 1
geometry_records = {name: {'cka': [], 'procrustes': [], 'principal_angle_mean': []} for name in geometry_conditions}

for layer in range(num_layers):
    h_base = collect_hidden_states(baseline_model, tokenizer, EVAL_CORPUS, layer, DEVICE)
    for name, model in geometry_conditions.items():
        h_cond = collect_hidden_states(model, tokenizer, EVAL_CORPUS, layer, DEVICE)
        cka = linear_cka(h_base, h_cond)
        proc = procrustes_distance(h_base, h_cond)
        angles = top_k_principal_angles(h_base, h_cond, k=10)
        geometry_records[name]['cka'].append(cka)
        geometry_records[name]['procrustes'].append(proc)
        geometry_records[name]['principal_angle_mean'].append(float(np.mean(angles)))

for name, rec in geometry_records.items():
    print(f"\n-- {name} --")
    for layer in range(num_layers):
        print(f"  layer {layer:2d}: CKA={rec['cka'][layer]:.4f}  "
              f"Procrustes={rec['procrustes'][layer]:.4f}  "
              f"mean principal angle={rec['principal_angle_mean'][layer]:6.2f} deg")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metric_titles = ['Linear CKA (vs. baseline)', 'Procrustes distance (vs. baseline)', 'Mean principal angle, top-10 (deg)']
metric_keys = ['cka', 'procrustes', 'principal_angle_mean']
cond_colors = {'aligned': COLORS['aligned'], 'unaligned': COLORS['unaligned'],
               'head_aligned': COLORS['head_aligned'], 'head_unaligned': COLORS['head_unaligned'],
               'global_rotation': COLORS['rotation_global']}

for ax, key, title in zip(axes, metric_keys, metric_titles):
    for name, rec in geometry_records.items():
        ax.plot(range(num_layers), rec[key], marker='o', markersize=3, label=name, color=cond_colors[name])
    ax.set_xlabel('Layer')
    ax.set_title(title)
axes[0].legend(fontsize=8)

plt.tight_layout()
plt.savefig('l1_section5_representation_geometry.png', dpi=150, bbox_inches='tight')
plt.show()


# harder tasks-do symmetry survive

In [ ]:
FEW_SHOT_ANTONYM = [
    ("hot -> cold\nbig -> small\nfast -> ", " slow"),
    ("up -> down\nleft -> right\nwet -> ", " dry"),
    ("happy -> sad\nlight -> dark\nyoung -> ", " old"),
    ("open -> closed\nfull -> empty\nloud -> ", " quiet"),
    ("day -> night\nhard -> soft\nrich -> ", " poor"),
]

ARITHMETIC = [
    ("2 + 3 =", " 5"),
    ("4 + 4 =", " 8"),
    ("7 - 2 =", " 5"),
    ("6 + 1 =", " 7"),
    ("9 - 3 =", " 6"),
    ("3 + 5 =", " 8"),
]

INDUCTION = [
    ("The cat sat on the mat. The dog sat on the", " mat"),
    ("John gave Mary a book. John gave Mary a", " book"),
    ("The red car and the red house and the red", " car"),
    ("A B C A B C A", " B"),
]

COMPOSITIONAL = [
    ("A robin is a bird. A bird is an animal. So a robin is an", " animal"),
    ("A rose is a flower. A flower is a plant. So a rose is a", " plant"),
    ("A trout is a fish. A fish is an animal. So a trout is an", " animal"),
    ("An oak is a tree. A tree is a plant. So an oak is a", " plant"),
]

MULTI_HOP = [
    ("Anna is Ben's mother. Ben is Cara's mother. So Anna is Cara's", " grandmother"),
    ("Tom is taller than Sam. Sam is taller than Lee. So Tom is taller than", " Lee"),
    ("The red box is inside the blue box. The blue box is inside the green box. "
     "So the red box is inside the", " green"),
    ("Monday comes before Tuesday. Tuesday comes before Wednesday. So Monday comes before", " Wednesday"),
]

HARD_TASKS = {
    'few_shot_antonym': FEW_SHOT_ANTONYM,
    'arithmetic': ARITHMETIC,
    'induction': INDUCTION,
    'compositional': COMPOSITIONAL,
    'multi_hop': MULTI_HOP,
}


def evaluate_completion_set(model, tokenizer, prompt_answer_pairs, device='cpu'):
    correct = 0
    model.eval()
    with torch.no_grad():
        for prompt, answer in prompt_answer_pairs:
            answer_id = tokenizer.encode(answer)[0]
            ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            pred_id = model(ids).logits[0, -1, :].argmax(dim=-1).item()
            correct += int(pred_id == answer_id)
    return correct / len(prompt_answer_pairs)


print("=" * 78)
print("SECTION 6: HARDER TASKS -- BASELINE vs. ALIGNED vs. UNALIGNED (MLP AND HEAD)")
print("=" * 78)

hard_task_conditions = {
    'baseline': baseline_model,
    'aligned (MLP)': aligned_model,
    'unaligned (MLP)': make_unaligned_permutation(baseline_model, TARGET_LAYER, seed=0, device=DEVICE)[0],
    'aligned (head)': head_aligned_model,
    'unaligned (head)': head_unaligned_model,
}

hard_task_results = {name: {} for name in hard_task_conditions}
for task_name, pairs in HARD_TASKS.items():
    print(f"\n-- {task_name} --")
    for cond_name, model in hard_task_conditions.items():
        acc = evaluate_completion_set(model, tokenizer, pairs, DEVICE)
        hard_task_results[cond_name][task_name] = acc
        print(f"  {cond_name:>18s}: {acc:.3f}")

fig, ax = plt.subplots(figsize=(12, 5))
task_names = list(HARD_TASKS.keys())
n_conditions = len(hard_task_conditions)
x = np.arange(len(task_names))
width = 0.8 / n_conditions
bar_colors_list = [COLORS['baseline'], COLORS['aligned'], COLORS['unaligned'], COLORS['head_aligned'], COLORS['head_unaligned']]

for i, (cond_name, accs) in enumerate(hard_task_results.items()):
    vals = [accs[t] for t in task_names]
    ax.bar(x + i * width - 0.4 + width / 2, vals, width, label=cond_name, color=bar_colors_list[i % len(bar_colors_list)])

ax.set_xticks(x)
ax.set_xticklabels(task_names, rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Harder-task accuracy across conditions')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('l1_section6_harder_tasks.png', dpi=150, bbox_inches='tight')
plt.show()


# stats

In [ ]:
def cohens_d(values):
    """One-sample Cohen's d against a null of 0 difference."""
    sd = values.std(ddof=1) if len(values) > 1 else 1e-8
    return float(values.mean() / (sd + 1e-8))


def mean_ci(values, confidence=0.95):
    mean = values.mean()
    sem = scipy_stats.sem(values) if len(values) > 1 else 0.0
    ci = scipy_stats.t.interval(confidence, len(values) - 1, loc=mean, scale=sem) if sem > 0 else (mean, mean)
    return float(mean), (float(ci[0]), float(ci[1]))


def seed_sweep_accuracy_diff(permute_fn, n_seeds, device='cpu'):
    diffs = []
    base_acc = next_token_accuracy(baseline_model, tokenizer, EVAL_CORPUS, device)
    for seed in range(n_seeds):
        m, _ = permute_fn(baseline_model, TARGET_LAYER, seed=seed, device=device)
        diffs.append(next_token_accuracy(m, tokenizer, EVAL_CORPUS, device) - base_acc)
    return np.array(diffs)


print("=" * 78)
print(f"SECTION 7: STATISTICAL VALIDATION (N={N_SEEDS} seeds, {int(CONFIDENCE*100)}% CI)")
print("=" * 78)

sweep_conditions = {
    'aligned (MLP)': lambda model, layer, seed, device: make_aligned_permutation(model, layer, seed=seed, device=device),
    'unaligned (MLP)': lambda model, layer, seed, device: make_unaligned_permutation(model, layer, seed=seed, device=device),
    'aligned (head)': lambda model, layer, seed, device: permute_attention_heads(model, layer, aligned=True, seed=seed, device=device),
    'unaligned (head)': lambda model, layer, seed, device: permute_attention_heads(model, layer, aligned=False, seed=seed, device=device),
}

stat_results = {}
for name, fn in sweep_conditions.items():
    diffs = seed_sweep_accuracy_diff(fn, N_SEEDS, DEVICE)
    mean, ci = mean_ci(diffs, CONFIDENCE)
    d = cohens_d(diffs)
    # Wilcoxon signed-rank against 0; falls back gracefully if all diffs are ~0 (no variance)
    try:
        stat, p_value = scipy_stats.wilcoxon(diffs)
    except ValueError:
        stat, p_value = float('nan'), 1.0 if np.allclose(diffs, 0) else float('nan')
    stat_results[name] = {'diffs': diffs, 'mean': mean, 'ci': ci, 'cohens_d': d, 'p_value': p_value}
    print(f"{name:>18s} | mean={mean:+.5f}  CI=[{ci[0]:+.5f}, {ci[1]:+.5f}]  "
          f"Cohen's d={d:+.3f}  Wilcoxon p={p_value:.4f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
names = list(stat_results.keys())
means = [stat_results[n]['mean'] for n in names]
ci_los = [stat_results[n]['ci'][0] for n in names]
ci_his = [stat_results[n]['ci'][1] for n in names]
errs = [[m - lo for m, lo in zip(means, ci_los)], [hi - m for m, hi in zip(means, ci_his)]]
point_colors = [COLORS['aligned'], COLORS['unaligned'], COLORS['head_aligned'], COLORS['head_unaligned']]

y_pos = np.arange(len(names))
ax.errorbar(means, y_pos, xerr=errs, fmt='o', capsize=4, color='black', ecolor='gray')
for i, c in enumerate(point_colors):
    ax.plot(means[i], y_pos[i], 'o', color=c, markersize=8)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Accuracy difference from baseline (mean, 95% CI)')
ax.set_title(f'Forest plot: accuracy diff across {N_SEEDS} seeds')
plt.tight_layout()
plt.savefig('l1_section7_statistical_validation.png', dpi=150, bbox_inches='tight')
plt.show()


# summary

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# Panel A: identity degradation curve (from Section 4)
ax_a = fig.add_subplot(gs[0, 0])
ax_a.errorbar(v_pct, acc_means, yerr=acc_stds, marker='o', color=COLORS['aligned'], capsize=3)
ax_a.axhline(baseline_result['accuracy'], color='gray', linestyle='--', linewidth=1)
ax_a.set_xlabel('Symmetry violation (%)')
ax_a.set_ylabel('Accuracy')
ax_a.set_title('A. Identity degradation curve')

# Panel B: CKA by layer across conditions (from Section 5)
ax_b = fig.add_subplot(gs[0, 1])
for name, rec in geometry_records.items():
    ax_b.plot(range(num_layers), rec['cka'], marker='o', markersize=3, label=name, color=cond_colors[name])
ax_b.set_xlabel('Layer')
ax_b.set_ylabel('Linear CKA vs. baseline')
ax_b.set_title('B. Representation geometry (CKA)')
ax_b.legend(fontsize=7)

# Panel C: harder-task accuracy grouped bars (from Section 6)
ax_c = fig.add_subplot(gs[1, 0])
for i, (cond_name, accs) in enumerate(hard_task_results.items()):
    vals = [accs[t] for t in task_names]
    ax_c.bar(x + i * width - 0.4 + width / 2, vals, width, label=cond_name, color=bar_colors_list[i % len(bar_colors_list)])
ax_c.set_xticks(x)
ax_c.set_xticklabels(task_names, rotation=20, ha='right', fontsize=8)
ax_c.set_ylabel('Accuracy')
ax_c.set_title('C. Harder-task accuracy')
ax_c.legend(fontsize=7)

# Panel D: statistical validation forest plot (from Section 7)
ax_d = fig.add_subplot(gs[1, 1])
ax_d.errorbar(means, y_pos, xerr=errs, fmt='o', capsize=4, color='black', ecolor='gray')
for i, c in enumerate(point_colors):
    ax_d.plot(means[i], y_pos[i], 'o', color=c, markersize=8)
ax_d.axvline(0, color='gray', linestyle='--', linewidth=1)
ax_d.set_yticks(y_pos)
ax_d.set_yticklabels(names, fontsize=8)
ax_d.set_xlabel('Accuracy diff from baseline')
ax_d.set_title(f'D. Statistical validation (N={N_SEEDS} seeds)')

fig.suptitle('WiringPrint L1: Internal Validation and Robustness -- Summary', fontsize=13, y=1.02)
plt.savefig('l1_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("Summary dashboard saved to l1_summary_dashboard.png")


# visualize findings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Finding A: MLP-unaligned vs. head-unaligned severity, normalized to baseline ---
metrics = ['Accuracy drop', 'Perplexity ratio\n(log10)', "|Cohen's d|\n(15-seed)"]
mlp_vals = [0.2746, np.log10(3892.797 / 56.628), 9.838]
head_vals = [0.0989, np.log10(314.338 / 56.628), 1.576]

x = np.arange(len(metrics))
width = 0.35
axes[0].bar(x - width/2, mlp_vals, width, label='MLP-unaligned', color='#d62728')
axes[0].bar(x + width/2, head_vals, width, label='Head-unaligned (V only)', color='#ff7f0e')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_title('Finding A: MLP-unaligned is consistently more\nsevere than head-unaligned')
axes[0].legend()

# --- Finding B: CKA vs. principal angle at layer 0, global rotation ---
metric_names = ['CKA', 'Procrustes\n(x100 for scale)', 'Principal angle / 90\n(normalized)']
layer0_vals = [1.0000, 0.0000 * 100, 84.59 / 90]
layer1_vals = [0.9447, 0.4390 * 100, 84.19 / 90]

x2 = np.arange(len(metric_names))
axes[1].bar(x2 - width/2, layer0_vals, width, label='Layer 0 (embedding, pre-nonlinearity)', color='#8c564b')
axes[1].bar(x2 + width/2, layer1_vals, width, label='Layer 1 (post block 0)', color='#9467bd')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metric_names)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Finding B: CKA/Procrustes miss the rotation at\nlayer 0; principal angle catches it (all normalized to [0,1])")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('l1_section10_findings.png', dpi=150, bbox_inches='tight')
plt.show()




# 3: generalization

In [ ]:

import copy
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.linalg import orthogonal_procrustes
from scipy.optimize import linear_sum_assignment
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = 'cuda'  # Changed from 'cpu' to 'cuda'
N_SEEDS = 15
CONFIDENCE = 0.95
TARGET_LAYER = 0

EVAL_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a subfield of artificial intelligence.",
    "Machine learning models learn patterns from large amounts of data.",
    "The transformer architecture relies on self-attention mechanisms.",
    "Scientists discovered a new species of frog in the rainforest.",
    "The stock market experienced significant volatility this week.",
    "Cooking pasta requires boiling water and a pinch of salt.",
    "The history of Rome spans over a thousand years.",
]

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})

ARCH_COLORS = {'gpt2': '#1f77b4', 'gpt2-medium': '#17becf', 'gpt2-large': '#2ca02c',
               'pythia': '#d62728', 'gpt-neo': '#9467bd'}


# =====================================================================================
# Generic loading
# =====================================================================================
def load_model(model_name, device='cpu', **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).to(device)
    model.eval()
    return model, tokenizer


# =====================================================================================
# Generic MLP permutation (handles Conv1D and nn.Linear)
# =====================================================================================
def find_mlp_layer(model, layer_idx):
    count = 0
    for name, module in model.named_modules():
        if hasattr(module, 'c_fc') and hasattr(module, 'c_proj'):
            is_conv1d = not isinstance(module.c_fc, torch.nn.Linear)
            if count == layer_idx:
                return name, module.c_fc, module.c_proj, is_conv1d
            count += 1
        elif hasattr(module, 'dense_h_to_4h') and hasattr(module, 'dense_4h_to_h'):
            if count == layer_idx:
                return name, module.dense_h_to_4h, module.dense_4h_to_h, False
            count += 1
    raise ValueError(f"No MLP layer found at index {layer_idx} for this architecture.")


def permute_mlp_generic(model, layer_idx, aligned=True, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    _, fc, proj, is_conv1d = find_mlp_layer(m, layer_idx)
    torch.manual_seed(seed)
    d_mlp = fc.nf if is_conv1d else fc.out_features
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device,dtype=fc.weight.data.dtype)
    P[torch.arange(d_mlp), perm] = 1.0
    with torch.no_grad():
        if is_conv1d:
            fc.weight.data = fc.weight.data @ P
            fc.bias.data = fc.bias.data @ P
            if aligned:
                proj.weight.data = P.T @ proj.weight.data
        else:
            fc.weight.data = P @ fc.weight.data       # nn.Linear: (out, in)
            fc.bias.data = P @ fc.bias.data
            if aligned:
                proj.weight.data = proj.weight.data @ P.T
    return m, perm


# =====================================================================================
# Generic attention-head permutation (handles 'gpt2', 'neox', 'separate' layouts)
# =====================================================================================
def detect_attn_kind(model, layer_idx):
    # GPT-2 style: transformer.h[i].attn.{c_attn, c_proj}
    try:
        attn = model.transformer.h[layer_idx].attn
        if hasattr(attn, 'c_attn') and hasattr(attn, 'c_proj'):
            return 'gpt2', attn
    except (AttributeError, IndexError):
        pass
    # GPT-NeoX / Pythia style: gpt_neox.layers[i].attention.{query_key_value, dense}
    try:
        attn = model.gpt_neox.layers[layer_idx].attention
        if hasattr(attn, 'query_key_value') and hasattr(attn, 'dense'):
            return 'neox', attn
    except (AttributeError, IndexError):
        pass
    # GPT-Neo style: transformer.h[i].attn.attention.{q_proj,k_proj,v_proj,out_proj}
    try:
        attn = model.transformer.h[layer_idx].attn.attention
        if hasattr(attn, 'q_proj') and hasattr(attn, 'out_proj'):
            return 'separate', attn
    except (AttributeError, IndexError):
        pass
    raise ValueError("Unrecognized attention module layout for this architecture.")


def build_head_block_permutation(num_heads, block_dim, perm, device='cpu'):
    """(num_heads*block_dim, num_heads*block_dim) permutation matrix moving contiguous
    block_dim-sized blocks according to perm (right-multiply convention: x @ P)."""
    total = num_heads * block_dim
    P = torch.zeros(total, total, device=device)
    for new_pos, old_head in enumerate(perm):
        rows = torch.arange(old_head * block_dim, (old_head + 1) * block_dim)
        cols = torch.arange(new_pos * block_dim, (new_pos + 1) * block_dim)
        P[rows, cols] = 1.0
    return P


def permute_attention_heads_generic(model, layer_idx=0, aligned=True, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    kind, attn = detect_attn_kind(m, layer_idx)
    torch.manual_seed(seed)

    if kind == 'gpt2':
        num_heads, embed_dim = attn.num_heads, attn.embed_dim
        head_dim = embed_dim // num_heads
        perm = torch.randperm(num_heads).tolist()
        P = build_head_block_permutation(num_heads, head_dim, perm, device=device)
        with torch.no_grad():
            w, b = attn.c_attn.weight.data, attn.c_attn.bias.data
            segments = [0, 1, 2] if aligned else [2]  # Q,K,V if aligned; V only if unaligned
            for seg in segments:
                sl = slice(seg * embed_dim, (seg + 1) * embed_dim)
                w[:, sl] = w[:, sl] @ P
                b[sl] = b[sl] @ P
            if aligned:
                attn.c_proj.weight.data = P.T @ attn.c_proj.weight.data

    elif kind == 'neox':
        num_heads, head_dim = attn.num_attention_heads, attn.head_size
        perm = torch.randperm(num_heads).tolist()
        P_qkv = build_head_block_permutation(num_heads, 3 * head_dim, perm, device=device)   # interleaved q,k,v per head
        P_out = build_head_block_permutation(num_heads, head_dim, perm, device=device)        # post-attention, per-head output
        with torch.no_grad():
            w, b = attn.query_key_value.weight.data, attn.query_key_value.bias.data
            if aligned:
                attn.query_key_value.weight.data = P_qkv.T @ w   # nn.Linear: rows=out
                attn.query_key_value.bias.data = P_qkv.T @ b
                attn.dense.weight.data = attn.dense.weight.data @ P_out
            else:
                # unaligned control: permute only the V third of each head's 3*head_dim block
                # (approximated here by permuting the full QKV block WITHOUT compensating
                # `dense` -- breaks the correspondence, the neox analogue of the V-only control)
                attn.query_key_value.weight.data = P_qkv.T @ w
                attn.query_key_value.bias.data = P_qkv.T @ b

    elif kind == 'separate':
        embed_dim = attn.q_proj.out_features
        num_heads = getattr(attn, 'num_heads', 12)
        head_dim = embed_dim // num_heads
        perm = torch.randperm(num_heads).tolist()
        P = build_head_block_permutation(num_heads, head_dim, perm, device=device)
        with torch.no_grad():
            projs = [attn.q_proj, attn.k_proj, attn.v_proj] if aligned else [attn.v_proj]
            for proj in projs:
                proj.weight.data = P.T @ proj.weight.data   # nn.Linear rows = out
                if proj.bias is not None:
                    proj.bias.data = P.T @ proj.bias.data
            if aligned:
                attn.out_proj.weight.data = attn.out_proj.weight.data @ P

    return m, perm


# =====================================================================================
# Core battery (unchanged from L0/L1)
# =====================================================================================
def next_token_accuracy(model, tokenizer, texts, device='cpu'):
    accs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits
            preds = logits[0, :-1, :].argmax(dim=-1)
            targets = ids[0, 1:]
            accs.append((preds == targets).float().mean().item())
    return float(np.mean(accs))


def perplexity(model, tokenizer, texts, device='cpu'):
    losses = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))


def run_battery(model, tokenizer, texts, device='cpu', label='model', verbose=True):
    acc = next_token_accuracy(model, tokenizer, texts, device)
    ppl = perplexity(model, tokenizer, texts, device)
    if verbose:
        print(f"{label:>28s} | accuracy={acc:.4f}  perplexity={ppl:10.3f}")
    return {'label': label, 'accuracy': acc, 'perplexity': ppl}


def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    num = ((X.T @ Y) ** 2).sum()
    den = np.sqrt(((X.T @ X) ** 2).sum() * ((Y.T @ Y) ** 2).sum())
    return float(num / den) if den > 0 else 0.0


def collect_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            rows.append(out.hidden_states[layer][0].cpu().numpy())
    return np.concatenate(rows, axis=0)  # (num_tokens, d_mlp)


print("Setup complete: generic multi-architecture toolkit loaded.")
print("Architectures supported: GPT-2 family ('gpt2'), Pythia/GPT-NeoX ('neox'), GPT-Neo ('separate').")

# scaling

In [ ]:
SCALING_MODELS = ['gpt2', 'gpt2-medium', 'gpt2-large']  # gpt2-xl omitted: 1.5B is likely
                                                          # infeasible on CPU in this pipeline;
                                                          # add it here if GPU is available.

print("=" * 78)
print("SECTION 2: SCALING STUDY -- GPT-2 SMALL / MEDIUM / LARGE")
print("=" * 78)

scaling_results = []
for model_name in SCALING_MODELS:
    print(f"\n-- {model_name} --")
    model, tokenizer = load_model(model_name, device=DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    n_layers = model.config.n_layer
    d_mlp = model.config.n_embd * 4

    baseline_res = run_battery(model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline')
    aligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
    aligned_res = run_battery(aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='aligned')
    unaligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)
    unaligned_res = run_battery(unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='unaligned')

    scaling_results.append({
        'model_name': model_name, 'n_params': n_params, 'n_layers': n_layers, 'd_mlp': d_mlp,
        'baseline_acc': baseline_res['accuracy'], 'aligned_acc': aligned_res['accuracy'],
        'unaligned_acc': unaligned_res['accuracy'],
        'baseline_ppl': baseline_res['perplexity'], 'aligned_ppl': aligned_res['perplexity'],
        'unaligned_ppl': unaligned_res['perplexity'],
    })
    del model, aligned_model, unaligned_model  # free memory before loading the next size

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
xs = [r['n_params'] / 1e6 for r in scaling_results]
axes[0].plot(xs, [r['baseline_acc'] for r in scaling_results], 'o-', label='baseline', color='#2b2b2b')
axes[0].plot(xs, [r['aligned_acc'] for r in scaling_results], 's--', label='aligned', color='#1f77b4')
axes[0].plot(xs, [r['unaligned_acc'] for r in scaling_results], '^--', label='unaligned', color='#d62728')
axes[0].set_xlabel('Parameters (millions)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs. model size')
axes[0].legend()

axes[1].plot(xs, np.log10([r['baseline_ppl'] for r in scaling_results]), 'o-', label='baseline', color='#2b2b2b')
axes[1].plot(xs, np.log10([r['aligned_ppl'] for r in scaling_results]), 's--', label='aligned', color='#1f77b4')
axes[1].plot(xs, np.log10([r['unaligned_ppl'] for r in scaling_results]), '^--', label='unaligned', color='#d62728')
axes[1].set_xlabel('Parameters (millions)')
axes[1].set_ylabel('log10(Perplexity)')
axes[1].set_title('Perplexity vs. model size')
axes[1].legend()

plt.tight_layout()
plt.savefig('l2_section2_scaling_study.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nScaling summary:")
for r in scaling_results:
    print(f"{r['model_name']:>14s} ({r['n_params']/1e6:6.1f}M params, {r['n_layers']:2d} layers, "
          f"d_mlp={r['d_mlp']:5d}): aligned diff={r['aligned_acc']-r['baseline_acc']:+.5f}  "
          f"unaligned diff={r['unaligned_acc']-r['baseline_acc']:+.5f}")


# cross-arch

In [ ]:
CROSS_ARCH_MODELS = {
    'gpt2': 'gpt2',
    'pythia': 'EleutherAI/pythia-160m',
    'gpt-neo': 'EleutherAI/gpt-neo-125m',
}

print("=" * 78)
print("SECTION 3: CROSS-ARCHITECTURE STUDY -- MLP PERMUTATION")
print("=" * 78)

cross_arch_results = {}
for family, model_name in CROSS_ARCH_MODELS.items():
    print(f"\n-- {family} ({model_name}) --")
    model, tokenizer = load_model(model_name, device=DEVICE)

    baseline_res = run_battery(model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline')
    aligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
    aligned_res = run_battery(aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='aligned MLP')
    unaligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)
    unaligned_res = run_battery(unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='unaligned MLP')

    cross_arch_results[family] = {'baseline': baseline_res, 'aligned': aligned_res, 'unaligned': unaligned_res}

print("\n" + "=" * 78)
print("SECTION 3b: CROSS-ARCHITECTURE STUDY -- ATTENTION-HEAD PERMUTATION")
print("=" * 78)

for family, model_name in CROSS_ARCH_MODELS.items():
    print(f"\n-- {family} ({model_name}) --")
    model, tokenizer = load_model(model_name, device=DEVICE)
    try:
        head_aligned_model, _ = permute_attention_heads_generic(model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
        head_aligned_res = run_battery(head_aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='head-aligned')
        head_unaligned_model, _ = permute_attention_heads_generic(model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)
        head_unaligned_res = run_battery(head_unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='head-unaligned')
        cross_arch_results[family]['head_aligned'] = head_aligned_res
        cross_arch_results[family]['head_unaligned'] = head_unaligned_res
    except (ValueError, AttributeError, RuntimeError) as e:
        print(f"  head permutation not implemented/failed for this architecture: {e}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
families = list(cross_arch_results.keys())
x = np.arange(len(families))
width = 0.25

for i, (cond, color) in enumerate([('baseline', '#2b2b2b'), ('aligned', '#1f77b4'), ('unaligned', '#d62728')]):
    vals = [cross_arch_results[f][cond]['accuracy'] for f in families]
    axes[0].bar(x + (i - 1) * width, vals, width, label=cond, color=color)
axes[0].set_xticks(x)
axes[0].set_xticklabels(families)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('MLP permutation across architectures')
axes[0].legend()

for i, (cond, color) in enumerate([('baseline', '#2b2b2b'), ('aligned', '#1f77b4'), ('unaligned', '#d62728')]):
    vals = [np.log10(cross_arch_results[f][cond]['perplexity']) for f in families]
    axes[1].bar(x + (i - 1) * width, vals, width, label=cond, color=color)
axes[1].set_xticks(x)
axes[1].set_xticklabels(families)
axes[1].set_ylabel('log10(Perplexity)')
axes[1].set_title('MLP permutation across architectures (perplexity)')
axes[1].legend()

plt.tight_layout()
plt.savefig('l2_section3_cross_architecture.png', dpi=150, bbox_inches='tight')
plt.show()


# training-time

In [ ]:
PYTHIA_MODEL = 'EleutherAI/pythia-160m'
PYTHIA_CHECKPOINTS = ['step1', 'step1000', 'step8000', 'step32000', 'step71000', 'step143000']

print("=" * 78)
print("SECTION 4: TRAINING-TIME STUDY -- PYTHIA CHECKPOINTS")
print("=" * 78)

checkpoint_records = []
for revision in PYTHIA_CHECKPOINTS:
    print(f"\n-- {revision} --")
    try:
        model, tokenizer = load_model(PYTHIA_MODEL, device=DEVICE, revision=revision)
    except Exception as e:
        print(f"  could not load checkpoint {revision}: {e}")
        continue

    baseline_res = run_battery(model, tokenizer, EVAL_CORPUS, DEVICE, label='baseline')
    aligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
    aligned_res = run_battery(aligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='aligned')
    unaligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)
    unaligned_res = run_battery(unaligned_model, tokenizer, EVAL_CORPUS, DEVICE, label='unaligned')

    h_base = collect_hidden_states(model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)
    h_aligned = collect_hidden_states(aligned_model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)
    h_unaligned = collect_hidden_states(unaligned_model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)
    cka_aligned = linear_cka(h_base, h_aligned)
    cka_unaligned = linear_cka(h_base, h_unaligned)

    checkpoint_records.append({
        'revision': revision, 'baseline_acc': baseline_res['accuracy'],
        'aligned_acc': aligned_res['accuracy'], 'unaligned_acc': unaligned_res['accuracy'],
        'cka_aligned': cka_aligned, 'cka_unaligned': cka_unaligned,
    })
    print(f"  CKA(baseline, aligned)={cka_aligned:.4f}   CKA(baseline, unaligned)={cka_unaligned:.4f}")
    del model, aligned_model, unaligned_model

if checkpoint_records:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    revs = [r['revision'] for r in checkpoint_records]
    x = np.arange(len(revs))

    axes[0].plot(x, [r['baseline_acc'] for r in checkpoint_records], 'o-', color='#2b2b2b', label='baseline')
    axes[0].plot(x, [r['aligned_acc'] for r in checkpoint_records], 's--', color='#1f77b4', label='aligned')
    axes[0].plot(x, [r['unaligned_acc'] for r in checkpoint_records], '^--', color='#d62728', label='unaligned')
    axes[0].set_xticks(x); axes[0].set_xticklabels(revs, rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('Accuracy'); axes[0].set_title('Accuracy over training'); axes[0].legend(fontsize=8)

    axes[1].plot(x, [r['cka_aligned'] for r in checkpoint_records], 's--', color='#1f77b4', label='CKA(aligned)')
    axes[1].plot(x, [r['cka_unaligned'] for r in checkpoint_records], '^--', color='#d62728', label='CKA(unaligned)')
    axes[1].set_xticks(x); axes[1].set_xticklabels(revs, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('CKA vs. baseline'); axes[1].set_title('Representation similarity over training'); axes[1].legend(fontsize=8)

    gap = [r['baseline_acc'] - r['unaligned_acc'] for r in checkpoint_records]
    axes[2].plot(x, gap, 'o-', color='#ff7f0e')
    axes[2].set_xticks(x); axes[2].set_xticklabels(revs, rotation=45, ha='right', fontsize=8)
    axes[2].set_ylabel('Baseline - unaligned accuracy gap')
    axes[2].set_title('Behavioral-stability gap over training')

    plt.tight_layout()
    plt.savefig('l2_section4_training_trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("\nNo checkpoints loaded successfully -- skipping trajectory plot.")


# cross-model

In [ ]:
def collect_mlp_hidden_activations(model, tokenizer, texts, layer_idx, device='cpu'):
    """Returns (num_tokens, d_mlp): the MLP hidden-layer activation (after c_fc, before
    c_proj) pooled across all tokens in `texts`, used as each neuron's 'signature'."""
    _, fc, _, is_conv1d = find_mlp_layer(model, layer_idx)
    activations = []

    def hook(module, inp, out):
        activations.append(out.detach().cpu().numpy())

    h = fc.register_forward_hook(hook)
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            activations.clear()
            model(ids)
            rows.append(activations[0][0])  # (seq_len, d_mlp)
    h.remove()
    return np.concatenate(rows, axis=0)  # (num_tokens, d_mlp)


def cosine_similarity_matrix(A, B):
    """A: (n_a, d_mlp_a) activations->neuron signatures as columns; here we treat each
    NEURON as a vector of its activations across tokens, so transpose first."""
    An = A / (np.linalg.norm(A, axis=0, keepdims=True) + 1e-8)
    Bn = B / (np.linalg.norm(B, axis=0, keepdims=True) + 1e-8)
    return An.T @ Bn  # (d_mlp_a, d_mlp_b)


def greedy_cosine_matching(sim_matrix):
    n_a, n_b = sim_matrix.shape
    matched_b = set()
    matching = -np.ones(n_a, dtype=int)
    order = np.dstack(np.unravel_index(np.argsort(-sim_matrix, axis=None), sim_matrix.shape))[0]
    for i, j in order:
        if matching[i] == -1 and j not in matched_b:
            matching[i] = j
            matched_b.add(j)
        if (matching != -1).all():
            break
    return matching


def hungarian_matching(sim_matrix):
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)  # maximize similarity
    matching = -np.ones(sim_matrix.shape[0], dtype=int)
    matching[row_ind] = col_ind
    return matching


print("=" * 78)
print("SECTION 5: CROSS-MODEL ALIGNMENT -- GPT2 <-> DISTILGPT2")
print("=" * 78)

model_a, tok_a = load_model('gpt2', device=DEVICE)
model_b, tok_b = load_model('distilgpt2', device=DEVICE)

acts_a = collect_mlp_hidden_activations(model_a, tok_a, EVAL_CORPUS, TARGET_LAYER, DEVICE)
acts_b = collect_mlp_hidden_activations(model_b, tok_b, EVAL_CORPUS, TARGET_LAYER, DEVICE)
print(f"gpt2 layer-0 MLP activations: {acts_a.shape}, distilgpt2: {acts_b.shape}")

sim_cosine = cosine_similarity_matrix(acts_a, acts_b)
greedy_match = greedy_cosine_matching(sim_cosine)
hungarian_match = hungarian_matching(sim_cosine)

greedy_score = float(np.mean([sim_cosine[i, greedy_match[i]] for i in range(len(greedy_match))]))
hungarian_score = float(np.mean([sim_cosine[i, hungarian_match[i]] for i in range(len(hungarian_match))]))
random_perm = np.random.RandomState(0).permutation(acts_b.shape[1])[:acts_a.shape[1]]
random_score = float(np.mean([sim_cosine[i, random_perm[i]] for i in range(len(random_perm))]))

print(f"\nMean matched cosine similarity:")
print(f"  random pairing:      {random_score:.4f}")
print(f"  greedy cosine match:  {greedy_score:.4f}")
print(f"  Hungarian (optimal):  {hungarian_score:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im0 = axes[0].imshow(sim_cosine[:60, :60], cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('Neuron-neuron cosine similarity\n(gpt2 vs. distilgpt2, first 60x60 shown)')
axes[0].set_xlabel('distilgpt2 neuron index')
axes[0].set_ylabel('gpt2 neuron index')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].bar(['random', 'greedy cosine', 'Hungarian'], [random_score, greedy_score, hungarian_score],
            color=['#2b2b2b', '#ff7f0e', '#1f77b4'])
axes[1].set_ylabel('Mean matched cosine similarity')
axes[1].set_title('Matching quality comparison')

plt.tight_layout()
plt.savefig('l2_section5_cross_model_alignment.png', dpi=150, bbox_inches='tight')
plt.show()


# Long context eval

In [ ]:
FILLER_SENTENCES = [
    "The weather today is mild with a light breeze from the west.",
    "Stock markets closed slightly higher after a quiet trading session.",
    "The museum's new exhibit features sculptures from the 19th century.",
    "Traffic on the highway was lighter than usual this morning.",
    "The committee will reconvene next Tuesday to finalize the budget.",
    "Local bakeries reported strong sales over the holiday weekend.",
    "The river's water level has risen slightly after recent rainfall.",
    "A new coffee shop opened downtown near the old train station.",
]

NEEDLE_TEMPLATES = [
    ("The secret code word is banana.", "What is the secret code word? The secret code word is", " banana"),
    ("The meeting is scheduled for Thursday at noon.", "What day is the meeting scheduled for? The meeting is scheduled for", " Thursday"),
    ("The package was shipped from Denver.", "Which city was the package shipped from? The package was shipped from", " Denver"),
]

LONG_MULTI_HOP_TEMPLATES = [
    ("Anna is Ben's mother.", "Ben is Cara's mother.", "So Anna is Cara's", " grandmother"),
    ("Tom is taller than Sam.", "Sam is taller than Lee.", "So Tom is taller than", " Lee"),
]


def build_haystack_prompt(needle_sentence, query, n_filler=6, seed=0):
    rng = np.random.RandomState(seed)
    fillers = list(np.array(FILLER_SENTENCES)[rng.choice(len(FILLER_SENTENCES), size=n_filler, replace=True)])
    insert_at = n_filler // 3
    context = fillers[:insert_at] + [needle_sentence] + fillers[insert_at:]
    return " ".join(context) + " " + query


def build_long_multihop_prompt(fact1, fact2, query, n_filler=4, seed=0):
    rng = np.random.RandomState(seed)
    fillers = list(np.array(FILLER_SENTENCES)[rng.choice(len(FILLER_SENTENCES), size=n_filler, replace=True)])
    context = [fact1] + fillers[:n_filler // 2] + [fact2] + fillers[n_filler // 2:]
    return " ".join(context) + " " + query


def evaluate_completion_set(model, tokenizer, prompt_answer_pairs, device='cpu'):
    correct = 0
    model.eval()
    with torch.no_grad():
        for prompt, answer in prompt_answer_pairs:
            answer_id = tokenizer.encode(answer)[0]
            ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            pred_id = model(ids).logits[0, -1, :].argmax(dim=-1).item()
            correct += int(pred_id == answer_id)
    return correct / len(prompt_answer_pairs)


def build_long_context_eval_set(n_filler=6):
    needle_pairs = [(build_haystack_prompt(n, q, n_filler=n_filler), a) for n, q, a in NEEDLE_TEMPLATES]
    hop_pairs = [(build_long_multihop_prompt(f1, f2, q, n_filler=n_filler), a) for f1, f2, q, a in LONG_MULTI_HOP_TEMPLATES]
    return needle_pairs, hop_pairs


print("=" * 78)
print("SECTION 6: LONG-CONTEXT EVALUATION -- NEEDLE-IN-HAYSTACK AND LONG MULTI-HOP")
print("=" * 78)

needle_pairs, hop_pairs = build_long_context_eval_set(n_filler=6)

long_context_model, long_context_tok = load_model('gpt2', device=DEVICE)
long_context_conditions = {
    'baseline': long_context_model,
    'aligned (MLP)': permute_mlp_generic(long_context_model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)[0],
    'unaligned (MLP)': permute_mlp_generic(long_context_model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)[0],
    'aligned (head)': permute_attention_heads_generic(long_context_model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)[0],
    'unaligned (head)': permute_attention_heads_generic(long_context_model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)[0],
}

long_context_results = {}
for name, model in long_context_conditions.items():
    needle_acc = evaluate_completion_set(model, long_context_tok, needle_pairs, DEVICE)
    hop_acc = evaluate_completion_set(model, long_context_tok, hop_pairs, DEVICE)
    long_context_results[name] = {'needle': needle_acc, 'long_multi_hop': hop_acc}
    print(f"{name:>18s} | needle-in-haystack={needle_acc:.3f}  long multi-hop={hop_acc:.3f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
names = list(long_context_results.keys())
x = np.arange(2)
width = 0.15
for i, name in enumerate(names):
    vals = [long_context_results[name]['needle'], long_context_results[name]['long_multi_hop']]
    ax.bar(x + i * width - 0.35, vals, width, label=name)
ax.set_xticks(x)
ax.set_xticklabels(['Needle-in-haystack', 'Long multi-hop'])
ax.set_ylabel('Accuracy')
ax.set_title('Long-context task accuracy across conditions')
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('l2_section6_long_context.png', dpi=150, bbox_inches='tight')
plt.show()


# causality

In [ ]:
INDUCTION_PROBE_PROMPTS = [
    ("The cat sat on the mat. The dog sat on the", " mat"),
    ("John gave Mary a book. John gave Mary a", " book"),
    ("A B C A B C A", " B"),
]

FACTUAL_CIRCUIT_PAIRS = [
    ("The capital of France is", " Paris", "The capital of Germany is", " Berlin"),
    ("The opposite of hot is", " cold", "The opposite of up is", " down"),
    ("A dog says", " woof", "A cat says", " meow"),
    ("The largest planet is", " Jupiter", "The smallest planet is", " Mercury"),
    ("A group of lions is called a", " pride", "A group of wolves is called a", " pack"),
    ("The freezing point of water is", " zero", "The boiling point of water is", " hundred"),
]

REASONING_PROBE_EXAMPLES = [
    ("Tom is taller than Sam", 1),
    ("The mountain is taller than the hill", 1),
    ("The oak is taller than the shrub", 1),
    ("Sam is shorter than Tom", 0),
    ("The hill is shorter than the mountain", 0),
    ("The shrub is shorter than the oak", 0),
]


def logit_diff(logits, correct_id, wrong_id):
    return (logits[0, -1, correct_id] - logits[0, -1, wrong_id]).item()


def run_causal_tracing(model, tokenizer, pairs, device='cpu'):
    n_layers = model.config.n_layer
    restoration_by_layer = np.zeros(n_layers)
    used_pairs = 0
    for clean_prompt, correct_ans, corrupted_prompt, wrong_ans in pairs:
        correct_id, wrong_id = tokenizer.encode(correct_ans)[0], tokenizer.encode(wrong_ans)[0]
        clean_ids = tokenizer(clean_prompt, return_tensors='pt').input_ids.to(device)
        corrupt_ids = tokenizer(corrupted_prompt, return_tensors='pt').input_ids.to(device)
        if clean_ids.shape[1] != corrupt_ids.shape[1]:
            continue
        used_pairs += 1
        with torch.no_grad():
            clean_out = model(clean_ids, output_hidden_states=True)
            clean_cache, clean_logits = clean_out.hidden_states, clean_out.logits
            corrupt_logits = model(corrupt_ids).logits
        clean_diff = logit_diff(clean_logits, correct_id, wrong_id)
        corrupt_diff = logit_diff(corrupt_logits, correct_id, wrong_id)
        subject_pos = clean_ids.shape[1] - 2
        for layer in range(n_layers):
            def hook(module, inp, out, layer=layer):
                out = list(out) if isinstance(out, tuple) else [out]
                out[0][:, subject_pos, :] = clean_cache[layer + 1][:, subject_pos, :]
                return tuple(out) if isinstance(out, tuple) else out[0]
            h = model.transformer.h[layer].register_forward_hook(hook)
            with torch.no_grad():
                patched_logits = model(corrupt_ids).logits
            h.remove()
            patched_diff = logit_diff(patched_logits, correct_id, wrong_id)
            denom = (clean_diff - corrupt_diff) if (clean_diff - corrupt_diff) != 0 else 1e-8
            restoration_by_layer[layer] += (patched_diff - corrupt_diff) / denom
    if used_pairs > 0:
        restoration_by_layer /= used_pairs
    return restoration_by_layer


print("=" * 78)
print("SECTION 7: EXPANDED CAUSAL ANALYSIS -- GPT-2-SMALL")
print("=" * 78)

causal_model, causal_tok = load_model('gpt2', device=DEVICE)
causal_aligned, _ = permute_mlp_generic(causal_model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)

print("\n-- Factual circuits, baseline vs. aligned --")
restoration_baseline = run_causal_tracing(causal_model, causal_tok, FACTUAL_CIRCUIT_PAIRS, DEVICE)
restoration_aligned = run_causal_tracing(causal_aligned, causal_tok, FACTUAL_CIRCUIT_PAIRS, DEVICE)
for layer in range(len(restoration_baseline)):
    print(f"  layer {layer:2d}: baseline restoration={restoration_baseline[layer]:.3f}  "
          f"aligned restoration={restoration_aligned[layer]:.3f}")

print("\n-- Reasoning-feature linear probing (comparative relation), baseline vs. aligned --")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

def probe_feature(model, tokenizer, examples, layer, device='cpu'):
    X, y = [], []
    with torch.no_grad():
        for text, label in examples:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            hs = model(ids, output_hidden_states=True).hidden_states[layer][0]
            X.append(hs.mean(dim=0).cpu().numpy())
            y.append(label)
    X, y = np.array(X), np.array(y)
    n_splits = min(3, min(np.bincount(y)))
    if n_splits < 2:
        return None
    clf = LogisticRegression(max_iter=1000)
    scores = cross_val_score(clf, X, y, cv=StratifiedKFold(n_splits=n_splits))
    return float(scores.mean())

probe_layer = 6
probe_baseline = probe_feature(causal_model, causal_tok, REASONING_PROBE_EXAMPLES, probe_layer, DEVICE)
probe_aligned = probe_feature(causal_aligned, causal_tok, REASONING_PROBE_EXAMPLES, probe_layer, DEVICE)
print(f"  comparative-relation probe (layer {probe_layer}): baseline={probe_baseline}  aligned={probe_aligned}")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(len(restoration_baseline)), restoration_baseline, 'o-', label='baseline', color='#2b2b2b')
ax.plot(range(len(restoration_aligned)), restoration_aligned, 's--', label='aligned', color='#1f77b4')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean restoration')
ax.set_title('Factual-circuit causal restoration: baseline vs. aligned permutation')
ax.legend()
plt.tight_layout()
plt.savefig('l2_section7_causal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


# stats

In [ ]:
def cohens_d(values):
    sd = values.std(ddof=1) if len(values) > 1 else 1e-8
    return float(values.mean() / (sd + 1e-8))


def mean_ci(values, confidence=0.95):
    mean = values.mean()
    sem = scipy_stats.sem(values) if len(values) > 1 else 0.0
    ci = scipy_stats.t.interval(confidence, len(values) - 1, loc=mean, scale=sem) if sem > 0 else (mean, mean)
    return float(mean), (float(ci[0]), float(ci[1]))


def seed_sweep_unaligned_diff(model_name, n_seeds, device='cpu'):
    model, tokenizer = load_model(model_name, device=device)
    base_acc = next_token_accuracy(model, tokenizer, EVAL_CORPUS, device)
    diffs = []
    for seed in range(n_seeds):
        m, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=False, seed=seed, device=device)
        diffs.append(next_token_accuracy(m, tokenizer, EVAL_CORPUS, device) - base_acc)
    return np.array(diffs)


print("=" * 78)
print(f"SECTION 8: STATISTICAL AGGREGATION -- META-ANALYSIS ACROSS ARCHITECTURES (N={N_SEEDS} seeds each)")
print("=" * 78)

META_MODELS = {'gpt2': 'gpt2', 'gpt2-medium': 'gpt2-medium', 'pythia': 'EleutherAI/pythia-160m', 'gpt-neo': 'EleutherAI/gpt-neo-125m'}

meta_records = []
for family, model_name in META_MODELS.items():
    print(f"\n-- {family} --")
    diffs = seed_sweep_unaligned_diff(model_name, N_SEEDS, DEVICE)
    mean, ci = mean_ci(diffs, CONFIDENCE)
    d = cohens_d(diffs)
    try:
        stat, p_value = scipy_stats.wilcoxon(diffs)
    except ValueError:
        stat, p_value = float('nan'), float('nan')
    meta_records.append({'family': family, 'mean': mean, 'ci': ci, 'cohens_d': d, 'p_value': p_value, 'diffs': diffs})
    print(f"  mean={mean:+.5f}  CI=[{ci[0]:+.5f}, {ci[1]:+.5f}]  Cohen's d={d:+.3f}  Wilcoxon p={p_value:.4f}")

d_values = np.array([r['cohens_d'] for r in meta_records])
print(f"\nMeta-analytic summary across {len(meta_records)} architectures:")
print(f"  mean Cohen's d = {d_values.mean():+.3f}  (cross-architecture SD = {d_values.std(ddof=1):.3f})")
print(f"  min/max Cohen's d = {d_values.min():+.3f} / {d_values.max():+.3f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
families = [r['family'] for r in meta_records]
means = [r['mean'] for r in meta_records]
ci_los = [r['ci'][0] for r in meta_records]
ci_his = [r['ci'][1] for r in meta_records]
errs = [[m - lo for m, lo in zip(means, ci_los)], [hi - m for m, hi in zip(means, ci_his)]]
y_pos = np.arange(len(families))

ax.errorbar(means, y_pos, xerr=errs, fmt='o', capsize=4, color='#d62728', ecolor='gray')
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.axvline(d_values.mean() and np.mean(means), color='#d62728', linestyle=':', linewidth=1, alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(families)
ax.set_xlabel('Unaligned-permutation accuracy diff (mean, 95% CI)')
ax.set_title(f'Meta-analysis forest plot: unaligned collapse across architectures (N={N_SEEDS} seeds each)')
plt.tight_layout()
plt.savefig('l2_section8_meta_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


# summary

In [ ]:
fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.3)

# Panel A: scaling curve (Section 2)
ax_a = fig.add_subplot(gs[0, 0])
xs = [r['n_params'] / 1e6 for r in scaling_results]
ax_a.plot(xs, [r['baseline_acc'] for r in scaling_results], 'o-', label='baseline', color='#2b2b2b')
ax_a.plot(xs, [r['aligned_acc'] for r in scaling_results], 's--', label='aligned', color='#1f77b4')
ax_a.plot(xs, [r['unaligned_acc'] for r in scaling_results], '^--', label='unaligned', color='#d62728')
ax_a.set_xlabel('Parameters (millions)')
ax_a.set_ylabel('Accuracy')
ax_a.set_title('A. Scaling study (GPT-2 Small/Medium/Large)')
ax_a.legend(fontsize=8)

# Panel B: training-time trajectory (Section 4)
ax_b = fig.add_subplot(gs[0, 1])
if checkpoint_records:
    revs = [r['revision'] for r in checkpoint_records]
    xb = np.arange(len(revs))
    ax_b.plot(xb, [r['baseline_acc'] for r in checkpoint_records], 'o-', color='#2b2b2b', label='baseline')
    ax_b.plot(xb, [r['aligned_acc'] for r in checkpoint_records], 's--', color='#1f77b4', label='aligned')
    ax_b.plot(xb, [r['unaligned_acc'] for r in checkpoint_records], '^--', color='#d62728', label='unaligned')
    ax_b.set_xticks(xb); ax_b.set_xticklabels(revs, rotation=45, ha='right', fontsize=7)
    ax_b.legend(fontsize=8)
ax_b.set_ylabel('Accuracy')
ax_b.set_title('B. Training-time trajectory (Pythia checkpoints)')

# Panel C: cross-model alignment heatmap (Section 5)
ax_c = fig.add_subplot(gs[1, 0])
im = ax_c.imshow(sim_cosine[:60, :60], cmap='RdBu_r', vmin=-1, vmax=1)
ax_c.set_xlabel('distilgpt2 neuron index')
ax_c.set_ylabel('gpt2 neuron index')
ax_c.set_title('C. Cross-model neuron similarity\n(gpt2 vs. distilgpt2)')
plt.colorbar(im, ax=ax_c, fraction=0.046)

# Panel D: cross-architecture meta-analysis forest plot (Section 8)
ax_d = fig.add_subplot(gs[1, 1])
families_d = [r['family'] for r in meta_records]
means_d = [r['mean'] for r in meta_records]
ci_los_d = [r['ci'][0] for r in meta_records]
ci_his_d = [r['ci'][1] for r in meta_records]
errs_d = [[m - lo for m, lo in zip(means_d, ci_los_d)], [hi - m for m, hi in zip(means_d, ci_his_d)]]
y_pos_d = np.arange(len(families_d))
ax_d.errorbar(means_d, y_pos_d, xerr=errs_d, fmt='o', capsize=4, color='#d62728', ecolor='gray')
ax_d.axvline(0, color='gray', linestyle='--', linewidth=1)
ax_d.set_yticks(y_pos_d); ax_d.set_yticklabels(families_d, fontsize=8)
ax_d.set_xlabel('Unaligned accuracy diff')
ax_d.set_title(f'D. Cross-architecture meta-analysis (N={N_SEEDS} seeds)')

fig.suptitle('WiringPrint L2: Generalization Across Models -- Summary', fontsize=13, y=1.02)
plt.savefig('l2_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# --- performance summary table, printed ---
print("=" * 100)
print("PERFORMANCE SUMMARY TABLE")
print("=" * 100)
print(f"{'Model':>16s} | {'Params(M)':>10s} | {'Baseline acc':>12s} | {'Aligned diff':>12s} | {'Unaligned diff':>14s}")
for r in scaling_results:
    print(f"{r['model_name']:>16s} | {r['n_params']/1e6:10.1f} | {r['baseline_acc']:12.4f} | "
          f"{r['aligned_acc']-r['baseline_acc']:+12.5f} | {r['unaligned_acc']-r['baseline_acc']:+14.5f}")
for family, res in cross_arch_results.items():
    print(f"{family:>16s} | {'--':>10s} | {res['baseline']['accuracy']:12.4f} | "
          f"{res['aligned']['accuracy']-res['baseline']['accuracy']:+12.5f} | "
          f"{res['unaligned']['accuracy']-res['baseline']['accuracy']:+14.5f}")


# annex-robust head-count detection for GPT-NeoX/Pythia

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_name, device='cpu', **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).to(device)
    model.eval()
    return model, tokenizer

def get_neox_head_config(attn, model):
    """Robust head-count/head-dim lookup for GPT-NeoX-style attention modules.
    Prefers module attributes if present (covers older transformers versions where
    they existed directly on the module), falls back to the model config (stable
    across versions) otherwise."""
    num_heads = None
    for attr in ['num_attention_heads', 'num_heads']:
        if hasattr(attn, attr):
            num_heads = getattr(attn, attr)
            break
    if num_heads is None:
        num_heads = model.config.num_attention_heads

    head_dim = None
    for attr in ['head_size', 'head_dim']:
        if hasattr(attn, attr):
            head_dim = getattr(attn, attr)
            break
    if head_dim is None:
        head_dim = model.config.hidden_size // num_heads

    return int(num_heads), int(head_dim)


def build_head_block_permutation(num_heads, block_dim, perm, device='cpu', dtype=None):
    """(num_heads*block_dim, num_heads*block_dim) permutation matrix moving contiguous
    block_dim-sized blocks according to perm (right-multiply convention: x @ P)."""
    total = num_heads * block_dim
    P = torch.zeros(total, total, device=device, dtype=dtype)
    for new_pos, old_head in enumerate(perm):
        rows = torch.arange(old_head * block_dim, (old_head + 1) * block_dim)
        cols = torch.arange(new_pos * block_dim, (new_pos + 1) * block_dim)
        P[rows, cols] = 1.0
    return P

def permute_attention_heads_generic_fixed(model, layer_idx=0, aligned=True, seed=0, device='cpu'):
    """Same as permute_attention_heads_generic, with the 'neox' branch's head-count
    lookup replaced by get_neox_head_config (Fix 1). 'gpt2' and 'separate' branches
    are unchanged from the original (they worked correctly in the executed run)."""
    m = copy.deepcopy(model)
    kind, attn = detect_attn_kind(m, layer_idx)
    torch.manual_seed(seed)

    if kind == 'gpt2':
        num_heads, embed_dim = attn.num_heads, attn.embed_dim
        head_dim = embed_dim // num_heads
        perm = torch.randperm(num_heads).tolist()
        P = build_head_block_permutation(num_heads, head_dim, perm, device=device)
        with torch.no_grad():
            w, b = attn.c_attn.weight.data, attn.c_attn.bias.data
            segments = [0, 1, 2] if aligned else [2]
            for seg in segments:
                sl = slice(seg * embed_dim, (seg + 1) * embed_dim)
                w[:, sl] = w[:, sl] @ P
                b[sl] = b[sl] @ P
            if aligned:
                attn.c_proj.weight.data = P.T @ attn.c_proj.weight.data

    elif kind == 'neox':
        num_heads, head_dim = get_neox_head_config(attn, m)          # <-- FIX 1
        perm = torch.randperm(num_heads).tolist()

        w, b = attn.query_key_value.weight.data, attn.query_key_value.bias.data
        # Get dtype from model weights
        weight_dtype = w.dtype

        P_qkv = build_head_block_permutation(num_heads, 3 * head_dim, perm, device=device, dtype=weight_dtype)
        P_out = build_head_block_permutation(num_heads, head_dim, perm, device=device, dtype=weight_dtype)
        with torch.no_grad():
            if aligned:
                attn.query_key_value.weight.data = P_qkv.T @ w
                attn.query_key_value.bias.data = P_qkv.T @ b
                attn.dense.weight.data = attn.dense.weight.data @ P_out
            else:
                attn.query_key_value.weight.data = P_qkv.T @ w
                attn.query_key_value.bias.data = P_qkv.T @ b

    elif kind == 'separate':
        embed_dim = attn.q_proj.out_features
        num_heads = getattr(attn, 'num_heads', 12)
        head_dim = embed_dim // num_heads
        perm = torch.randperm(num_heads).tolist()
        P = build_head_block_permutation(num_heads, head_dim, perm, device=device)
        with torch.no_grad():
            projs = [attn.q_proj, attn.k_proj, attn.v_proj] if aligned else [attn.v_proj]
            for proj in projs:
                proj.weight.data = P.T @ proj.weight.data
                if proj.bias is not None:
                    proj.bias.data = P.T @ proj.bias.data
            if aligned:
                attn.out_proj.weight.data = attn.out_proj.weight.data @ P

    return m, perm


print("=" * 78)
print("SECTION 11.1: CORRECTED RE-RUN -- PYTHIA ATTENTION-HEAD PERMUTATION")
print("=" * 78)

pythia_model_fix, pythia_tok_fix = load_model('EleutherAI/pythia-160m', device=DEVICE)

pythia_head_aligned, _ = permute_attention_heads_generic_fixed(pythia_model_fix, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
pythia_head_unaligned, _ = permute_attention_heads_generic_fixed(pythia_model_fix, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)

pythia_baseline_res = run_battery(pythia_model_fix, pythia_tok_fix, EVAL_CORPUS, DEVICE, label='baseline')
pythia_head_aligned_res = run_battery(pythia_head_aligned, pythia_tok_fix, EVAL_CORPUS, DEVICE, label='head-aligned (fixed)')
pythia_head_unaligned_res = run_battery(pythia_head_unaligned, pythia_tok_fix, EVAL_CORPUS, DEVICE, label='head-unaligned (fixed)')

with torch.no_grad():
    sample_ids = pythia_tok_fix(EVAL_CORPUS[0], return_tensors='pt').input_ids.to(DEVICE)
    base_logits = pythia_model_fix(sample_ids).logits
    ha_diff = (base_logits - pythia_head_aligned(sample_ids).logits).abs().max().item()
    hu_diff = (base_logits - pythia_head_unaligned(sample_ids).logits).abs().max().item()

print(f"\nMax abs logit diff, head-aligned:   {ha_diff:.2e}  (expect ~1e-4, floating-point noise)")
print(f"Max abs logit diff, head-unaligned: {hu_diff:.3f}  (expect large)")

# update the cross-architecture results table from Section 3b with the corrected values
if 'cross_arch_results' in globals() and 'pythia' in cross_arch_results:
    cross_arch_results['pythia']['head_aligned'] = pythia_head_aligned_res
    cross_arch_results['pythia']['head_unaligned'] = pythia_head_unaligned_res
    print("\ncross_arch_results['pythia'] updated in place with the corrected head-permutation rows.")

In [ ]:
def linear_cka_stable(X, Y):
    """Numerically stable linear CKA (Fix 2): normalizes both inputs to unit Frobenius
    norm before computing cross terms (mathematically a no-op for CKA, since it is
    scale-invariant by construction) and computes in float64."""
    X = X.astype(np.float64)
    Y = Y.astype(np.float64)
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)

    X_norm = np.linalg.norm(X)
    Y_norm = np.linalg.norm(Y)
    if X_norm < 1e-12 or Y_norm < 1e-12:
        return 0.0
    Xn, Yn = X / X_norm, Y / Y_norm

    num = np.linalg.norm(Xn.T @ Yn) ** 2
    den = np.linalg.norm(Xn.T @ Xn) * np.linalg.norm(Yn.T @ Yn)
    return float(num / den) if den > 1e-12 else 0.0


print("=" * 78)
print("SECTION 11.2: CORRECTED RE-RUN -- PYTHIA TRAINING-TIME REPRESENTATION SIMILARITY")
print("=" * 78)

corrected_checkpoint_records = []
for revision in PYTHIA_CHECKPOINTS:
    print(f"\n-- {revision} --")
    try:
        model, tokenizer = load_model(PYTHIA_MODEL, device=DEVICE, revision=revision)
    except Exception as e:
        print(f"  could not load checkpoint {revision}: {e}")
        continue

    base_acc = next_token_accuracy(model, tokenizer, EVAL_CORPUS, DEVICE)
    aligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=True, seed=0, device=DEVICE)
    unaligned_model, _ = permute_mlp_generic(model, TARGET_LAYER, aligned=False, seed=0, device=DEVICE)

    h_base = collect_hidden_states(model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)
    h_aligned = collect_hidden_states(aligned_model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)
    h_unaligned = collect_hidden_states(unaligned_model, tokenizer, EVAL_CORPUS, layer=1, device=DEVICE)

    cka_aligned = linear_cka_stable(h_base, h_aligned)
    cka_unaligned = linear_cka_stable(h_base, h_unaligned)

    print(f"  hidden-state magnitude (mean abs): {np.abs(h_base).mean():.2e}  "
          f"(this is what overflowed float32's squared range in the original run)")
    print(f"  CKA(baseline, aligned)={cka_aligned:.4f}   CKA(baseline, unaligned)={cka_unaligned:.4f}")

    corrected_checkpoint_records.append({
        'revision': revision, 'baseline_acc': base_acc,
        'cka_aligned': cka_aligned, 'cka_unaligned': cka_unaligned,
    })
    del model, aligned_model, unaligned_model

if corrected_checkpoint_records:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    revs = [r['revision'] for r in corrected_checkpoint_records]
    x = np.arange(len(revs))
    ax.plot(x, [r['cka_aligned'] for r in corrected_checkpoint_records], 's--', color='#1f77b4', label='CKA(aligned)')
    ax.plot(x, [r['cka_unaligned'] for r in corrected_checkpoint_records], '^--', color='#d62728', label='CKA(unaligned)')
    ax.set_xticks(x); ax.set_xticklabels(revs, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('CKA vs. baseline')
    ax.set_title('Corrected representation similarity over training (Fix 2 applied)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('l2_section11_corrected_cka_trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()


# 4 - symmetry

In [ ]:

import copy
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.linalg import orthogonal_procrustes
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from transformers import GPT2Tokenizer, GPT2LMHeadModel

MODEL_NAME = 'gpt2'
TARGET_LAYER = 0
DEVICE = 'cpu'
N_SEEDS = 10

EVAL_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is a subfield of artificial intelligence.",
    "Machine learning models learn patterns from large amounts of data.",
    "The transformer architecture relies on self-attention mechanisms.",
    "Scientists discovered a new species of frog in the rainforest.",
    "The stock market experienced significant volatility this week.",
    "Cooking pasta requires boiling water and a pinch of salt.",
    "The history of Rome spans over a thousand years.",
]

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
baseline_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_model.eval()


def next_token_accuracy(model, tokenizer, texts, device='cpu'):
    accs = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            logits = model(ids).logits
            preds = logits[0, :-1, :].argmax(dim=-1)
            targets = ids[0, 1:]
            accs.append((preds == targets).float().mean().item())
    return float(np.mean(accs))


def perplexity(model, tokenizer, texts, device='cpu'):
    losses = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            if ids.shape[1] < 2:
                continue
            out = model(ids, labels=ids)
            losses.append(out.loss.item())
    return float(np.exp(np.mean(losses)))


def linear_cka_stable(X, Y):
    X = X.astype(np.float64); Y = Y.astype(np.float64)
    X = X - X.mean(axis=0, keepdims=True); Y = Y - Y.mean(axis=0, keepdims=True)
    X_norm, Y_norm = np.linalg.norm(X), np.linalg.norm(Y)
    if X_norm < 1e-12 or Y_norm < 1e-12:
        return 0.0
    Xn, Yn = X / X_norm, Y / Y_norm
    num = np.linalg.norm(Xn.T @ Yn) ** 2
    den = np.linalg.norm(Xn.T @ Xn) * np.linalg.norm(Yn.T @ Yn)
    return float(num / den) if den > 1e-12 else 0.0


def collect_hidden_states(model, tokenizer, texts, layer, device='cpu'):
    rows = []
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, return_tensors='pt').input_ids.to(device)
            out = model(ids, output_hidden_states=True)
            rows.append(out.hidden_states[layer][0].cpu().numpy())
    return np.concatenate(rows, axis=0)


CAUSAL_TRACE_PAIRS = [
    ("The capital of France is", " Paris", "The capital of Germany is", " Berlin"),
    ("The opposite of hot is", " cold", "The opposite of up is", " down"),
    ("A dog says", " woof", "A cat says", " meow"),
]


def logit_diff(logits, correct_id, wrong_id):
    return (logits[0, -1, correct_id] - logits[0, -1, wrong_id]).item()


def run_causal_tracing(model, tokenizer, pairs, device='cpu'):
    n_layers = model.config.n_layer
    restoration = np.zeros(n_layers)
    used = 0
    for clean_prompt, correct_ans, corrupted_prompt, wrong_ans in pairs:
        correct_id, wrong_id = tokenizer.encode(correct_ans)[0], tokenizer.encode(wrong_ans)[0]
        clean_ids = tokenizer(clean_prompt, return_tensors='pt').input_ids.to(device)
        corrupt_ids = tokenizer(corrupted_prompt, return_tensors='pt').input_ids.to(device)
        if clean_ids.shape[1] != corrupt_ids.shape[1]:
            continue
        used += 1
        with torch.no_grad():
            clean_out = model(clean_ids, output_hidden_states=True)
            clean_cache, clean_logits = clean_out.hidden_states, clean_out.logits
            corrupt_logits = model(corrupt_ids).logits
        clean_diff = logit_diff(clean_logits, correct_id, wrong_id)
        corrupt_diff = logit_diff(corrupt_logits, correct_id, wrong_id)
        subject_pos = clean_ids.shape[1] - 2
        for layer in range(n_layers):
            def hook(module, inp, out, layer=layer):
                out = list(out) if isinstance(out, tuple) else [out]
                out[0][:, subject_pos, :] = clean_cache[layer + 1][:, subject_pos, :]
                return tuple(out) if isinstance(out, tuple) else out[0]
            h = model.transformer.h[layer].register_forward_hook(hook)
            with torch.no_grad():
                patched_logits = model(corrupt_ids).logits
            h.remove()
            patched_diff = logit_diff(patched_logits, correct_id, wrong_id)
            denom = (clean_diff - corrupt_diff) if (clean_diff - corrupt_diff) != 0 else 1e-8
            restoration[layer] += (patched_diff - corrupt_diff) / denom
    if used > 0:
        restoration /= used
    return restoration


def evaluate_transformation(model, label, verbose=True):
    """The full battery: behavior, representation, causal -- applied identically to
    every transformation in the library for a fair, side-by-side comparison."""
    acc = next_token_accuracy(model, tokenizer, EVAL_CORPUS, DEVICE)
    ppl = perplexity(model, tokenizer, EVAL_CORPUS, DEVICE)

    num_layers = baseline_model.config.n_layer + 1
    ckas = []
    for layer in range(num_layers):
        h_base = collect_hidden_states(baseline_model, tokenizer, EVAL_CORPUS, layer, DEVICE)
        h_t = collect_hidden_states(model, tokenizer, EVAL_CORPUS, layer, DEVICE)
        ckas.append(linear_cka_stable(h_base, h_t))
    mean_cka = float(np.mean(ckas))
    final_cka = ckas[-1]

    restoration = run_causal_tracing(model, tokenizer, CAUSAL_TRACE_PAIRS, DEVICE)
    mean_restoration = float(np.mean(restoration))

    baseline_acc = next_token_accuracy(baseline_model, tokenizer, EVAL_CORPUS, DEVICE)
    acc_diff = acc - baseline_acc

    if verbose:
        print(f"{label:>32s} | acc={acc:.4f} (diff={acc_diff:+.4f})  ppl={ppl:10.2f}  "
              f"mean_CKA={mean_cka:.4f}  final_CKA={final_cka:.4f}  mean_restoration={mean_restoration:+.3f}")

    return {'label': label, 'accuracy': acc, 'acc_diff': acc_diff, 'perplexity': ppl,
            'mean_cka': mean_cka, 'final_cka': final_cka, 'per_layer_cka': ckas,
            'mean_restoration': mean_restoration, 'restoration_curve': restoration.tolist()}


baseline_summary = evaluate_transformation(baseline_model, 'baseline')
print("\nSetup complete: core battery loaded.")


In [ ]:
def get_mlp(model, layer_idx):
    return model.transformer.h[layer_idx].mlp


# --- 2.1 MLP neuron permutation (aligned) ---
def transform_mlp_permutation(model, magnitude=1.0, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    m = copy.deepcopy(model)
    mlp = get_mlp(m, layer_idx)
    torch.manual_seed(seed)
    d_mlp = mlp.c_fc.nf
    perm = torch.randperm(d_mlp)
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), perm] = 1.0
    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        mlp.c_proj.weight.data = P.T @ mlp.c_proj.weight.data
    return m


# --- 2.2 Attention-head permutation (aligned) ---
def build_head_block_permutation(num_heads, block_dim, perm, device='cpu'):
    total = num_heads * block_dim
    P = torch.zeros(total, total, device=device)
    for new_pos, old_head in enumerate(perm):
        rows = torch.arange(old_head * block_dim, (old_head + 1) * block_dim)
        cols = torch.arange(new_pos * block_dim, (new_pos + 1) * block_dim)
        P[rows, cols] = 1.0
    return P


def transform_head_permutation(model, magnitude=1.0, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    m = copy.deepcopy(model)
    attn = m.transformer.h[layer_idx].attn
    num_heads, embed_dim = attn.num_heads, attn.embed_dim
    head_dim = embed_dim // num_heads
    torch.manual_seed(seed)
    perm = torch.randperm(num_heads).tolist()
    P = build_head_block_permutation(num_heads, head_dim, perm, device=device)
    with torch.no_grad():
        w, b = attn.c_attn.weight.data, attn.c_attn.bias.data
        for seg in range(3):
            sl = slice(seg * embed_dim, (seg + 1) * embed_dim)
            w[:, sl] = w[:, sl] @ P
            b[sl] = b[sl] @ P
        attn.c_proj.weight.data = P.T @ attn.c_proj.weight.data
    return m


# --- 2.3 Multi-head block permutation ---
def transform_multihead_block_permutation(model, magnitude=2, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    """magnitude here is the block size k (number of heads per block)."""
    k = max(1, int(magnitude))
    m = copy.deepcopy(model)
    attn = m.transformer.h[layer_idx].attn
    num_heads, embed_dim = attn.num_heads, attn.embed_dim
    head_dim = embed_dim // num_heads
    n_blocks = num_heads // k
    torch.manual_seed(seed)
    block_perm = torch.randperm(n_blocks).tolist()
    # expand block permutation to a full head permutation (heads within a block keep order)
    head_perm = []
    for new_block_pos, old_block in enumerate(block_perm):
        head_perm.extend(range(old_block * k, old_block * k + k))
    # any leftover heads (if num_heads not divisible by k) stay in place
    head_perm.extend(range(n_blocks * k, num_heads))
    P = build_head_block_permutation(num_heads, head_dim, head_perm, device=device)
    with torch.no_grad():
        w, b = attn.c_attn.weight.data, attn.c_attn.bias.data
        for seg in range(3):
            sl = slice(seg * embed_dim, (seg + 1) * embed_dim)
            w[:, sl] = w[:, sl] @ P
            b[sl] = b[sl] @ P
        attn.c_proj.weight.data = P.T @ attn.c_proj.weight.data
    return m


# --- 2.4 Residual stream basis change (general invertible, non-orthogonal) ---
def transform_basis_change(model, magnitude=0.3, seed=0, device='cpu'):
    """magnitude = epsilon controlling how far A departs from the identity (and hence
    from orthogonality); A = I + epsilon * N for random N."""
    m = copy.deepcopy(model)
    n_embd = m.config.n_embd
    torch.manual_seed(seed)
    N = torch.randn(n_embd, n_embd, device=device)
    A = torch.eye(n_embd, device=device) + magnitude * N
    A_inv = torch.linalg.inv(A)
    with torch.no_grad():
        m.transformer.wte.weight.data = m.transformer.wte.weight.data @ A
        m.transformer.wpe.weight.data = m.transformer.wpe.weight.data @ A
        for block in m.transformer.h:
            block.attn.c_attn.weight.data = A_inv @ block.attn.c_attn.weight.data
            block.attn.c_proj.weight.data = block.attn.c_proj.weight.data @ A
            block.mlp.c_fc.weight.data = A_inv @ block.mlp.c_fc.weight.data
            block.mlp.c_proj.weight.data = block.mlp.c_proj.weight.data @ A
    return m


# --- 2.5 Orthogonal rotation (residual stream, global) ---
def transform_global_rotation(model, magnitude=1.0, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    n_embd = m.config.n_embd
    torch.manual_seed(seed)
    A = torch.randn(n_embd, n_embd, device=device)
    Q, _ = torch.linalg.qr(A)
    with torch.no_grad():
        m.transformer.wte.weight.data = m.transformer.wte.weight.data @ Q
        m.transformer.wpe.weight.data = m.transformer.wpe.weight.data @ Q
        for block in m.transformer.h:
            block.attn.c_attn.weight.data = Q.T @ block.attn.c_attn.weight.data
            block.attn.c_proj.weight.data = block.attn.c_proj.weight.data @ Q
            block.mlp.c_fc.weight.data = Q.T @ block.mlp.c_fc.weight.data
            block.mlp.c_proj.weight.data = block.mlp.c_proj.weight.data @ Q
    return m


print("Transformations 2.1-2.5 implemented.")


In [ ]:
# --- 2.6 Random Gaussian perturbation ---
def transform_gaussian_noise(model, magnitude=0.1, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    m = copy.deepcopy(model)
    mlp = get_mlp(m, layer_idx)
    torch.manual_seed(seed)
    with torch.no_grad():
        for w in [mlp.c_fc.weight.data, mlp.c_proj.weight.data]:
            sigma = magnitude * w.std()
            w.add_(torch.randn_like(w) * sigma)
    return m


# --- 2.7 Weight interpolation ---
def transform_weight_interpolation(model, magnitude=0.5, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    """magnitude = alpha in [0, 1]; interpolates between baseline and its own
    aligned-permutation image."""
    alpha = magnitude
    permuted = transform_mlp_permutation(model, seed=seed, layer_idx=layer_idx, device=device)
    m = copy.deepcopy(model)
    mlp_base, mlp_perm, mlp_m = get_mlp(model, layer_idx), get_mlp(permuted, layer_idx), get_mlp(m, layer_idx)
    with torch.no_grad():
        mlp_m.c_fc.weight.data = (1 - alpha) * mlp_base.c_fc.weight.data + alpha * mlp_perm.c_fc.weight.data
        mlp_m.c_fc.bias.data = (1 - alpha) * mlp_base.c_fc.bias.data + alpha * mlp_perm.c_fc.bias.data
        mlp_m.c_proj.weight.data = (1 - alpha) * mlp_base.c_proj.weight.data + alpha * mlp_perm.c_proj.weight.data
    return m


# --- 2.8 Low-rank perturbation ---
def transform_low_rank_perturbation(model, magnitude=4, seed=0, layer_idx=TARGET_LAYER, scale=1.0, device='cpu'):
    """magnitude = rank r of the additive perturbation U V^T (scaled by scale/sqrt(r))."""
    r = max(1, int(magnitude))
    m = copy.deepcopy(model)
    mlp = get_mlp(m, layer_idx)
    torch.manual_seed(seed)
    with torch.no_grad():
        for w in [mlp.c_fc.weight.data, mlp.c_proj.weight.data]:
            rows, cols = w.shape
            U = torch.randn(rows, r, device=device)
            V = torch.randn(cols, r, device=device)
            w.add_((scale / math.sqrt(r)) * (U @ V.T) * w.std())
    return m


# --- 2.9 Sparse neuron permutation ---
def transform_sparse_permutation(model, magnitude=0.1, seed=0, layer_idx=TARGET_LAYER, device='cpu'):
    """magnitude = fraction k/d_mlp of neurons involved in the (aligned) permutation;
    the rest are left as the identity map."""
    m = copy.deepcopy(model)
    mlp = get_mlp(m, layer_idx)
    d_mlp = mlp.c_fc.nf
    torch.manual_seed(seed)
    k = max(2, int(round(d_mlp * magnitude)))
    subset = torch.randperm(d_mlp)[:k]
    local_perm = subset[torch.randperm(k)]
    full_perm = torch.arange(d_mlp)
    full_perm[subset] = local_perm
    P = torch.zeros(d_mlp, d_mlp, device=device)
    P[torch.arange(d_mlp), full_perm] = 1.0
    with torch.no_grad():
        mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
        mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
        mlp.c_proj.weight.data = P.T @ mlp.c_proj.weight.data
    return m


# --- 2.10 Layer-local permutation (swept across which layer) ---
def transform_layer_local_permutation(model, magnitude=0, seed=0, device='cpu'):
    """magnitude = which layer index to permute."""
    layer_idx = int(magnitude)
    return transform_mlp_permutation(model, seed=seed, layer_idx=layer_idx, device=device)


# --- 2.11 Cross-layer permutation (independent permutations, every layer at once) ---
def transform_cross_layer_permutation(model, magnitude=1.0, seed=0, device='cpu'):
    m = copy.deepcopy(model)
    n_layers = m.config.n_layer
    for layer_idx in range(n_layers):
        mlp = get_mlp(m, layer_idx)
        torch.manual_seed(seed * 1000 + layer_idx)  # independent permutation per layer
        d_mlp = mlp.c_fc.nf
        perm = torch.randperm(d_mlp)
        P = torch.zeros(d_mlp, d_mlp, device=device)
        P[torch.arange(d_mlp), perm] = 1.0
        with torch.no_grad():
            mlp.c_fc.weight.data = mlp.c_fc.weight.data @ P
            mlp.c_fc.bias.data = mlp.c_fc.bias.data @ P
            mlp.c_proj.weight.data = P.T @ mlp.c_proj.weight.data
    return m


print("Transformations 2.6-2.11 implemented. Full library of 11 transformations ready.")

TRANSFORMATION_LIBRARY = {
    'mlp_permutation': (transform_mlp_permutation, 1.0),
    'head_permutation': (transform_head_permutation, 1.0),
    'multihead_block_permutation': (transform_multihead_block_permutation, 2),
    'basis_change': (transform_basis_change, 0.3),
    'global_rotation': (transform_global_rotation, 1.0),
    'gaussian_noise': (transform_gaussian_noise, 0.1),
    'weight_interpolation': (transform_weight_interpolation, 0.5),
    'low_rank_perturbation': (transform_low_rank_perturbation, 4),
    'sparse_permutation': (transform_sparse_permutation, 0.1),
    'layer_local_permutation': (transform_layer_local_permutation, 6),
    'cross_layer_permutation': (transform_cross_layer_permutation, 1.0),
}
print(f"\n{len(TRANSFORMATION_LIBRARY)} transformations registered:")
for name in TRANSFORMATION_LIBRARY:
    print(f"  - {name}")


# experimental

In [ ]:
print("=" * 100)
print("SECTION 3: EXPERIMENTAL PIPELINE -- FULL BATTERY ACROSS THE TRANSFORMATION LIBRARY")
print("=" * 100)

pipeline_results = {'baseline': baseline_summary}

for name, (fn, default_magnitude) in TRANSFORMATION_LIBRARY.items():
    transformed_model = fn(baseline_model, magnitude=default_magnitude, seed=0, device=DEVICE)
    result = evaluate_transformation(transformed_model, name)
    pipeline_results[name] = result
    del transformed_model

print("\nPipeline complete: all 11 transformations evaluated against the full battery.")


# identity preservation

In [ ]:
def classify_identity(acc_diff, mean_cka):
    if abs(acc_diff) < 0.01 and mean_cka > 0.99:
        return 'Exact'
    elif abs(acc_diff) < 0.15:
        return 'Approximate'
    else:
        return 'Broken'


print("=" * 100)
print("SECTION 4: IDENTITY PRESERVATION MATRIX")
print("=" * 100)
print(f"{'Transformation':>30s} | {'Acc diff':>9s} | {'Mean CKA':>9s} | {'Final CKA':>10s} | {'Restoration':>12s} | {'Class':>12s}")
print("-" * 100)

identity_matrix = []
for name, r in pipeline_results.items():
    if name == 'baseline':
        continue
    cls = classify_identity(r['acc_diff'], r['mean_cka'])
    identity_matrix.append({'name': name, 'acc_diff': r['acc_diff'], 'mean_cka': r['mean_cka'],
                             'final_cka': r['final_cka'], 'restoration': r['mean_restoration'], 'class': cls})
    print(f"{name:>30s} | {r['acc_diff']:+9.4f} | {r['mean_cka']:9.4f} | {r['final_cka']:10.4f} | "
          f"{r['mean_restoration']:+12.3f} | {cls:>12s}")

fig, ax = plt.subplots(figsize=(9, 6))
class_colors = {'Exact': '#2ca02c', 'Approximate': '#ff7f0e', 'Broken': '#d62728'}
names = [r['name'] for r in identity_matrix]
metrics = ['acc_diff', 'mean_cka', 'final_cka', 'restoration']
metric_labels = ['Accuracy diff', 'Mean CKA', 'Final-layer CKA', 'Causal restoration']
data = np.array([[r[m] for m in metrics] for r in identity_matrix])

# normalize each column to [0,1] for a comparable heatmap (direction-adjusted so
# "1.0 = most identity-preserving" everywhere)
norm_data = np.zeros_like(data)
norm_data[:, 0] = 1 - np.clip(np.abs(data[:, 0]) / (np.abs(data[:, 0]).max() + 1e-8), 0, 1)  # smaller |diff| is better
norm_data[:, 1] = data[:, 1]  # CKA already in [0,1], higher is better
norm_data[:, 2] = data[:, 2]
norm_data[:, 3] = np.clip(data[:, 3], 0, 1.2) / 1.2  # restoration near 1 is best

im = ax.imshow(norm_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(metric_labels))); ax.set_xticklabels(metric_labels, rotation=20, ha='right')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i, r in enumerate(identity_matrix):
    ax.text(-0.7, i, r['class'], va='center', ha='right', fontsize=9, fontweight='bold', color=class_colors[r['class']])
plt.colorbar(im, ax=ax, fraction=0.046, label='Normalized (1.0 = most identity-preserving)')
ax.set_title('Identity preservation matrix (color) with categorical class (left labels)')
plt.tight_layout()
plt.savefig('l25_section4_identity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


# boundary

In [ ]:
print("=" * 100)
print("SECTION 5: BOUNDARY ANALYSIS -- DEGRADATION CURVES")
print("=" * 100)

BOUNDARY_SWEEPS = {
    'gaussian_noise': (transform_gaussian_noise, [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.5]),
    'low_rank_perturbation': (transform_low_rank_perturbation, [0, 1, 2, 4, 8, 16, 32]),
    'weight_interpolation': (transform_weight_interpolation, [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]),
    'sparse_permutation': (transform_sparse_permutation, [0.0, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0]),
    'basis_change': (transform_basis_change, [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.5]),
}

boundary_curves = {}
for name, (fn, magnitudes) in BOUNDARY_SWEEPS.items():
    print(f"\n-- {name} --")
    accs = []
    for mag in magnitudes:
        if mag == 0.0:
            acc = baseline_summary['accuracy']
        else:
            m = fn(baseline_model, magnitude=mag, seed=0, device=DEVICE)
            acc = next_token_accuracy(m, tokenizer, EVAL_CORPUS, DEVICE)
            del m
        accs.append(acc)
        print(f"  magnitude={mag:6.2f} | accuracy={acc:.4f}")
    boundary_curves[name] = {'magnitudes': magnitudes, 'accuracies': accs}

fig, ax = plt.subplots(figsize=(10, 6))
markers = ['o', 's', '^', 'D', 'v']
for (name, curve), marker in zip(boundary_curves.items(), markers):
    # normalize each transformation's magnitude axis to [0,1] for overlay comparability
    mags = np.array(curve['magnitudes'], dtype=float)
    mags_norm = mags / (mags.max() if mags.max() > 0 else 1)
    ax.plot(mags_norm, curve['accuracies'], marker=marker, label=name)
ax.axhline(baseline_summary['accuracy'], color='gray', linestyle='--', linewidth=1, label='baseline')
ax.set_xlabel('Normalized transformation magnitude (0=none, 1=max tested)')
ax.set_ylabel('Accuracy')
ax.set_title('Identity degradation curves across transformation types')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('l25_section5_boundary_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


# discovery

In [ ]:
print("=" * 100)
print("SECTION 6: SYMMETRY DISCOVERY -- HIERARCHICAL CLUSTERING")
print("=" * 100)

feature_names = ['acc_diff', 'mean_cka', 'final_cka', 'restoration']
names_for_clustering = [r['name'] for r in identity_matrix]
X_features = np.array([[r[f] for f in feature_names] for r in identity_matrix])

# standardize each feature (zero mean, unit variance) so no single metric dominates
# the distance purely due to differences in scale (e.g. accuracy diff vs. CKA)
X_std = (X_features - X_features.mean(axis=0)) / (X_features.std(axis=0) + 1e-8)

Z = linkage(X_std, method='average', metric='euclidean')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
dend = dendrogram(Z, labels=names_for_clustering, ax=axes[0], leaf_rotation=45, leaf_font_size=9)
axes[0].set_title('Hierarchical clustering dendrogram\n(behavioral + representational + causal features)')
axes[0].set_ylabel('Distance (standardized Euclidean, average linkage)')

# cut the dendrogram into k clusters and report cluster membership alongside the
# Section 4 categorical class, to check whether they agree
K_CLUSTERS = 3
cluster_labels = fcluster(Z, K_CLUSTERS, criterion='maxclust')
print(f"\nCluster assignment (k={K_CLUSTERS}) vs. Section 4 categorical class:")
print(f"{'Transformation':>30s} | {'Cluster':>7s} | {'Section 4 class':>16s}")
for name, cluster, r in zip(names_for_clustering, cluster_labels, identity_matrix):
    print(f"{name:>30s} | {cluster:7d} | {r['class']:>16s}")

order = dend['leaves']
dist_matrix = squareform(np.round(
    np.array([[np.linalg.norm(X_std[i] - X_std[j]) for j in range(len(X_std))] for i in range(len(X_std))]), 6
))
dist_square = np.array([[np.linalg.norm(X_std[i] - X_std[j]) for j in range(len(X_std))] for i in range(len(X_std))])
im = axes[1].imshow(dist_square[np.ix_(order, order)], cmap='viridis_r')
axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels([names_for_clustering[i] for i in order], rotation=90, fontsize=8)
axes[1].set_yticks(range(len(order))); axes[1].set_yticklabels([names_for_clustering[i] for i in order], fontsize=8)
axes[1].set_title('Pairwise transformation-distance matrix\n(reordered by dendrogram)')
plt.colorbar(im, ax=axes[1], fraction=0.046, label='Standardized Euclidean distance')

plt.tight_layout()
plt.savefig('l25_section6_symmetry_discovery.png', dpi=150, bbox_inches='tight')
plt.show()


# summary

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.3)

# Panel A: identity preservation heatmap (Section 4)
ax_a = fig.add_subplot(gs[0, 0])
im_a = ax_a.imshow(norm_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax_a.set_xticks(range(len(metric_labels))); ax_a.set_xticklabels(metric_labels, rotation=20, ha='right', fontsize=8)
ax_a.set_yticks(range(len(names))); ax_a.set_yticklabels(names, fontsize=8)
ax_a.set_title('A. Identity preservation matrix')

# Panel B: degradation curves (Section 5)
ax_b = fig.add_subplot(gs[0, 1])
for (name, curve), marker in zip(boundary_curves.items(), markers):
    mags = np.array(curve['magnitudes'], dtype=float)
    mags_norm = mags / (mags.max() if mags.max() > 0 else 1)
    ax_b.plot(mags_norm, curve['accuracies'], marker=marker, label=name, markersize=4)
ax_b.axhline(baseline_summary['accuracy'], color='gray', linestyle='--', linewidth=1)
ax_b.set_xlabel('Normalized magnitude'); ax_b.set_ylabel('Accuracy')
ax_b.set_title('B. Degradation curves')
ax_b.legend(fontsize=6)

# Panel C: clustering dendrogram (Section 6)
ax_c = fig.add_subplot(gs[1, 0])
dendrogram(Z, labels=names_for_clustering, ax=ax_c, leaf_rotation=45, leaf_font_size=7)
ax_c.set_title('C. Transformation clustering')
ax_c.set_ylabel('Distance')

# Panel D: symmetry containment chain (S_n subset O(n) subset GL(n)), transformations placed by group
ax_d = fig.add_subplot(gs[1, 1])
group_membership = {
    'mlp_permutation': 'S_n', 'head_permutation': 'S_n', 'multihead_block_permutation': 'S_n',
    'sparse_permutation': 'S_n', 'layer_local_permutation': 'S_n', 'cross_layer_permutation': 'S_n',
    'global_rotation': 'O(n)',
    'basis_change': 'GL(n)', 'weight_interpolation': 'GL(n)',
    'gaussian_noise': 'none', 'low_rank_perturbation': 'none',
}
group_x = {'S_n': 0, 'O(n)': 1, 'GL(n)': 2, 'none': 3}
class_colors = {'Exact': '#2ca02c', 'Approximate': '#ff7f0e', 'Broken': '#d62728'}
name_to_class = {r['name']: r['class'] for r in identity_matrix}

for name, group in group_membership.items():
    x = group_x[group]
    y = hash(name) % 100 / 100.0  # arbitrary vertical jitter for visual separation
    cls = name_to_class.get(name, 'Broken')
    ax_d.scatter(x, y, s=250, color=class_colors[cls], edgecolor='black', zorder=3)
    ax_d.annotate(name, (x, y), textcoords="offset points", xytext=(8, 0), fontsize=7, va='center')

ax_d.set_xticks([0, 1, 2, 3])
ax_d.set_xticklabels(['$S_n$\n(permutations)', '$O(n)$\n(orthogonal)', '$GL(n)$\n(invertible)', 'no group\nstructure'])
ax_d.set_yticks([])
ax_d.set_xlim(-0.5, 4.5)
ax_d.set_title('D. Transformations along the symmetry containment chain\n(color = empirical class)')
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=l)
                   for l, c in class_colors.items()]
ax_d.legend(handles=legend_handles, loc='upper right', fontsize=8)

fig.suptitle('WiringPrint L2.5: Characterizing the Symmetry Landscape -- Summary', fontsize=13, y=1.02)
plt.savefig('l25_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("Summary dashboard saved to l25_summary_dashboard.png")


#mathematics

In [ ]:
import math
import itertools
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.stats import ortho_group

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})

D_MODEL, D_MLP = 4, 16

W_fc = torch.randn(D_MODEL, D_MLP)
b_fc = torch.randn(D_MLP)
W_proj = torch.randn(D_MLP, D_MODEL)

def mlp_block(x, W_fc, b_fc, W_proj):
    """A single transformer MLP block: fc -> GELU -> proj, exactly GPT-2's structure
    with `Conv1D`-convention row-vector matmul (x @ W)."""
    h = F.gelu(x @ W_fc + b_fc)
    return h @ W_proj

def layernorm(x, gamma=None, beta=None, eps=1e-5):
    """Standard LayerNorm: per-row mean/variance over the feature axis, elementwise
    affine. gamma/beta default to ones/zeros (unlearned) unless supplied."""
    if gamma is None:
        gamma = torch.ones(x.shape[-1])
    if beta is None:
        beta = torch.zeros(x.shape[-1])
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta

x_test = torch.randn(5, D_MODEL)  # 5 example "tokens"
print("Synthetic validation model ready: W_fc", tuple(W_fc.shape), "W_proj", tuple(W_proj.shape))
print("Test input batch:", tuple(x_test.shape))
print("\nBaseline MLP output (first row):", mlp_block(x_test, W_fc, b_fc, W_proj)[0].detach().numpy().round(4))


In [ ]:
import itertools

print("=" * 78)
print("SECTION 3 VALIDATION: GROUP ACTION AXIOMS AND ORBIT SIZE (toy model, d_mlp=4 for enumerability)")
print("=" * 78)

# small enough d_mlp to enumerate the FULL orbit (4! = 24 elements) exhaustively
D_MLP_SMALL = 4
torch.manual_seed(1)
W_fc_small = torch.randn(D_MODEL, D_MLP_SMALL)
b_fc_small = torch.randn(D_MLP_SMALL)
W_proj_small = torch.randn(D_MLP_SMALL, D_MODEL)

def perm_matrix(perm, device='cpu'):
    n = len(perm)
    P = torch.zeros(n, n, device=device)
    P[torch.arange(n), torch.tensor(perm)] = 1.0
    return P

def apply_permutation(W_fc, b_fc, W_proj, perm):
    P = perm_matrix(perm)
    return W_fc @ P, b_fc @ P, P.T @ W_proj

# --- validate identity element acts trivially ---
identity_perm = list(range(D_MLP_SMALL))
W_fc_e, b_fc_e, W_proj_e = apply_permutation(W_fc_small, b_fc_small, W_proj_small, identity_perm)
print(f"Identity acts trivially: {torch.allclose(W_fc_e, W_fc_small)} "
      f"(max diff {torch.abs(W_fc_e - W_fc_small).max().item():.2e})")

# --- validate compatibility with composition: phi(g1, phi(g2, theta)) == phi(g1*g2, theta) ---
perm1 = [1, 0, 2, 3]
perm2 = [0, 2, 3, 1]
# apply perm2 then perm1 (function composition order matching phi(g1, phi(g2, theta)))
Wfc_2, bfc_2, Wproj_2 = apply_permutation(W_fc_small, b_fc_small, W_proj_small, perm2)
Wfc_12, bfc_12, Wproj_12 = apply_permutation(Wfc_2, bfc_2, Wproj_2, perm1)
# composed permutation g1 . g2 (apply perm2's relabeling, then perm1's, as a single permutation of indices)
composed_perm = [perm1[perm2[i]] for i in range(D_MLP_SMALL)]
Wfc_direct, bfc_direct, Wproj_direct = apply_permutation(W_fc_small, b_fc_small, W_proj_small, composed_perm)

print(f"\nCompatibility with composition holds: {torch.allclose(Wfc_12, Wfc_direct)} "
      f"(max diff {torch.abs(Wfc_12 - Wfc_direct).max().item():.2e})")

# --- enumerate the FULL orbit and check its size matches d_mlp! (generic weights => trivial stabilizer) ---
all_perms = list(itertools.permutations(range(D_MLP_SMALL)))
orbit_tensors = []
for perm in all_perms:
    Wfc_p, bfc_p, Wproj_p = apply_permutation(W_fc_small, b_fc_small, W_proj_small, list(perm))
    orbit_tensors.append(torch.cat([Wfc_p.flatten(), bfc_p.flatten(), Wproj_p.flatten()]))

orbit_tensors = torch.stack(orbit_tensors)
# count numerically distinct orbit elements (pairwise distances all > tolerance => trivial stabilizer)
pairwise_dists = torch.cdist(orbit_tensors, orbit_tensors)
n_distinct = (pairwise_dists.fill_diagonal_(float('inf')) > 1e-6).all(dim=1).sum().item()

print(f"\n|G| = {D_MLP_SMALL}! = {math.factorial(D_MLP_SMALL)}")
print(f"Enumerated orbit size: {len(all_perms)}")
print(f"Numerically distinct orbit elements: {n_distinct} / {len(all_perms)}")
print(f"Trivial stabilizer confirmed: {n_distinct == math.factorial(D_MLP_SMALL)}  "
      f"(i.e., |Stab(theta)|=1, matching |Orb|*|Stab|=|G| via the orbit-stabilizer theorem)")


# analytical

In [ ]:
print("=" * 78)
print("SECTION 4 VALIDATION: THEOREMS 4.2, 4.3, AND LEMMA 4.5 / COROLLARY 4.6")
print("=" * 78)

torch.manual_seed(2)
perm = torch.randperm(D_MLP)
P_full = torch.zeros(D_MLP, D_MLP)
P_full[torch.arange(D_MLP), perm] = 1.0

# --- Theorem 4.2: aligned permutation is exact, checked on EVERY row of a fresh batch ---
W_fc_aligned = W_fc @ P_full
b_fc_aligned = b_fc @ P_full
W_proj_aligned = P_full.T @ W_proj

out_baseline = mlp_block(x_test, W_fc, b_fc, W_proj)
out_aligned = mlp_block(x_test, W_fc_aligned, b_fc_aligned, W_proj_aligned)
max_diff_aligned = (out_baseline - out_aligned).abs().max().item()
print(f"\nTheorem 4.2 (aligned permutation): max |diff| over batch = {max_diff_aligned:.2e}  "
      f"(machine-precision zero, exactly as proven)")

# --- Theorem 4.3: unaligned (uncompensated) permutation generically breaks equality ---
W_proj_unaligned = W_proj  # NOT compensated
out_unaligned = mlp_block(x_test, W_fc_aligned, b_fc_aligned, W_proj_unaligned)
max_diff_unaligned = (out_baseline - out_unaligned).abs().max().item()
print(f"Theorem 4.3 (unaligned permutation): max |diff| over batch = {max_diff_unaligned:.4f}  "
      f"(generically nonzero, as proven -- would be 0 only on a measure-zero set of W_proj)")

# --- Lemma 4.5: permutation preserves per-row mean/variance exactly; rotation does not ---
v = torch.randn(1, 8)
v_permuted = v @ perm_matrix(torch.randperm(8).tolist())
Q_random, _ = torch.linalg.qr(torch.randn(8, 8))
v_rotated = v @ Q_random

mean_orig, var_orig = v.mean().item(), v.var(unbiased=False).item()
mean_perm, var_perm = v_permuted.mean().item(), v_permuted.var(unbiased=False).item()
mean_rot, var_rot = v_rotated.mean().item(), v_rotated.var(unbiased=False).item()

print(f"\nLemma 4.5 -- mean:  original={mean_orig:.6f}  permuted={mean_perm:.6f} (diff={abs(mean_orig-mean_perm):.2e})  "
      f"rotated={mean_rot:.6f} (diff={abs(mean_orig-mean_rot):.4f})")
print(f"Lemma 4.5 -- var:   original={var_orig:.6f}  permuted={var_perm:.6f} (diff={abs(var_orig-var_perm):.2e})  "
      f"rotated={var_rot:.6f} (diff={abs(var_orig-var_rot):.4f})")

# --- Corollary 4.6: LayerNorm commutes with permutation (gamma,beta relabeled together),
#     but not with rotation (no relabeling of elementwise gamma,beta can repair it) ---
gamma = torch.randn(8).abs() + 0.5
beta = torch.randn(8)
perm8 = torch.randperm(8)
P8 = perm_matrix(perm8.tolist())

ln_orig = layernorm(v, gamma, beta)
ln_orig_then_permute = ln_orig @ P8
v_perm8 = v @ P8
gamma_perm, beta_perm = gamma @ P8, beta @ P8
ln_of_permuted = layernorm(v_perm8, gamma_perm, beta_perm)
diff_ln_perm = (ln_orig_then_permute - ln_of_permuted).abs().max().item()

v_rot8 = v @ Q_random
ln_orig_then_rotate = ln_orig @ Q_random
ln_of_rotated_same_params = layernorm(v_rot8, gamma, beta)  # no relabeling possible for a generic rotation
diff_ln_rot = (ln_orig_then_rotate - ln_of_rotated_same_params).abs().max().item()

print(f"\nCorollary 4.6 -- LN(xP) vs. LN(x)P (gamma,beta relabeled together): max diff = {diff_ln_perm:.2e}  (commutes exactly)")
print(f"Corollary 4.6 -- LN(xQ) vs. LN(x)Q (no valid relabeling exists):      max diff = {diff_ln_rot:.4f}  (does NOT commute)")


# symmetry continuity

In [ ]:
print("=" * 78)
print("SECTION 5 VALIDATION: WEIGHT INTERPOLATION BARRIER (toy model, reproducing L2.5's qualitative result)")
print("=" * 78)

W_fc_perm, b_fc_perm, W_proj_perm = W_fc @ P_full, b_fc @ P_full, P_full.T @ W_proj

alphas = np.linspace(0, 1, 21)
outputs_norm_diff = []
for alpha in alphas:
    W_fc_interp = (1 - alpha) * W_fc + alpha * W_fc_perm
    b_fc_interp = (1 - alpha) * b_fc + alpha * b_fc_perm
    W_proj_interp = (1 - alpha) * W_proj + alpha * W_proj_perm
    out_interp = mlp_block(x_test, W_fc_interp, b_fc_interp, W_proj_interp)
    diff = (out_interp - out_baseline).norm().item()
    outputs_norm_diff.append(diff)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(alphas, outputs_norm_diff, 'o-', color='#d62728')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel(r'Interpolation coefficient $\alpha$')
ax.set_ylabel(r'$\|\mathrm{MLP}_{\theta(\alpha)}(x) - \mathrm{MLP}_\theta(x)\|$')
ax.set_title('Toy-model interpolation barrier between $\\theta$ and $P\\theta$\n(both endpoints exact, interior diverges -- reproduces L2.5\'s qualitative finding)')
plt.tight_layout()
plt.savefig('l3_section5_interpolation_barrier.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOutput deviation at alpha=0.0: {outputs_norm_diff[0]:.2e}  (exact, per Theorem 4.2)")
print(f"Output deviation at alpha=0.5: {outputs_norm_diff[10]:.4f}  (interior -- generically nonzero)")
print(f"Output deviation at alpha=1.0: {outputs_norm_diff[-1]:.2e}  (exact, per Theorem 4.2)")
print("\nThis toy-model curve is qualitatively the same shape as L2.5's real GPT-2 result")
print("(accuracy 0.3155 -> 0.135 (worst, near alpha=0.4) -> 0.3155): endpoints exact, interior barrier.")


# cross-model equivalence

In [ ]:
print("=" * 78)
print("SECTION 6 VALIDATION: THEOREM 6.2 (EXACT RECOVERY, PLANTED CASE) AND")
print("COROLLARY 6.3 (APPROXIMATE RECOVERY, INDEPENDENT CASE)")
print("=" * 78)

torch.manual_seed(3)
N_SAMPLES, D_HIDDEN = 200, 32

# --- Planted case: A2 is EXACTLY A1 permuted (Theorem 6.2) ---
A1 = torch.randn(N_SAMPLES, D_HIDDEN)
true_perm = torch.randperm(D_HIDDEN)
P_true = torch.zeros(D_HIDDEN, D_HIDDEN)
P_true[torch.arange(D_HIDDEN), true_perm] = 1.0
A2_planted = A1 @ P_true  # EXACT planted correspondence, no noise

cost_matrix_planted = -(A1.T @ A2_planted).numpy()  # negative similarity for minimization
row_ind, col_ind = linear_sum_assignment(cost_matrix_planted)
recovered_perm = col_ind  # hat P as an index array
exact_recovery = np.array_equal(recovered_perm, true_perm.numpy())
print(f"\nPlanted case: exact recovery of true permutation = {exact_recovery}")
print(f"(Hungarian recovers all {D_HIDDEN} correspondences correctly: {exact_recovery}, as Theorem 6.2 predicts)")

# --- Independent case: A2 comes from a DIFFERENT random matrix entirely (no planted P*) ---
A2_independent = torch.randn(N_SAMPLES, D_HIDDEN)  # unrelated to A1
cost_matrix_indep = -(A1.T @ A2_independent).numpy()
row_ind_i, col_ind_i = linear_sum_assignment(cost_matrix_indep)
mean_sim_indep = -cost_matrix_indep[row_ind_i, col_ind_i].mean() / N_SAMPLES

# random-pairing baseline for comparison
random_perm_baseline = np.random.RandomState(0).permutation(D_HIDDEN)
mean_sim_random = -cost_matrix_indep[np.arange(D_HIDDEN), random_perm_baseline].mean() / N_SAMPLES

print(f"\nIndependent case (no true correspondence exists):")
print(f"  Hungarian-matched mean similarity:  {mean_sim_indep:.4f}")
print(f"  Random-pairing mean similarity:     {mean_sim_random:.4f}")
print(f"  (Hungarian still finds the BEST available pairing, per Corollary 6.3, but this")
print(f"   is evidence of approximate structure, not recovery of a ground-truth P*)")


# visualization summary of math behind transformers

In [ ]:
fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.35)

# Panel A: Cayley graph of S_3 (orbit of a 3-neuron example under all 6 permutations)
ax_a = fig.add_subplot(gs[0, 0])
import itertools as it
s3_perms = list(it.permutations([0, 1, 2]))
angles = np.linspace(0, 2 * np.pi, len(s3_perms), endpoint=False)
positions = {p: (np.cos(a), np.sin(a)) for p, a in zip(s3_perms, angles)}
# connect permutations differing by a single transposition (adjacent transpositions generate S_3)
generators = [(0, 1), (1, 2)]
for p in s3_perms:
    for i, j in generators:
        q = list(p)
        q[i], q[j] = q[j], q[i]
        q = tuple(q)
        x1, y1 = positions[p]; x2, y2 = positions[q]
        ax_a.plot([x1, x2], [y1, y2], color='gray', alpha=0.5, zorder=1)
for p, (x, y) in positions.items():
    ax_a.scatter(x, y, s=800, color='#2ca02c', edgecolor='black', zorder=2)
    ax_a.annotate(str(p), (x, y), ha='center', va='center', fontsize=8, zorder=3)
ax_a.set_xlim(-1.4, 1.4); ax_a.set_ylim(-1.4, 1.4)
ax_a.set_xticks([]); ax_a.set_yticks([])
ax_a.set_title('A. Cayley graph of $S_3$: full orbit of a 3-neuron layer\n(all 6 nodes compute the identical function, Theorem 4.2)')

# Panel B: quotient space schematic
ax_b = fig.add_subplot(gs[0, 1])
np.random.seed(7)
orbit_colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e']
for oc_idx, color in enumerate(orbit_colors):
    cx, cy = np.random.uniform(0.2, 0.8, 2)
    orbit_pts = [(cx + 0.06 * np.cos(t), cy + 0.06 * np.sin(t)) for t in np.linspace(0, 2 * np.pi, 5, endpoint=False)]
    for (x, y) in orbit_pts:
        ax_b.scatter(x, y, s=60, color=color, alpha=0.7)
    ax_b.scatter(cx, cy + 0.35, s=200, color=color, marker='s', edgecolor='black', zorder=3)
    ax_b.annotate('', xy=(cx, cy + 0.28), xytext=(cx, cy + 0.08),
                  arrowprops=dict(arrowstyle='->', color=color, alpha=0.6))
ax_b.text(0.5, 1.05, r'$\Theta$ (parameter space): each cluster = one orbit', ha='center', fontsize=9, transform=ax_b.transAxes)
ax_b.text(0.5, -0.08, r'$\Theta/G$ (quotient space): each orbit collapses to one point', ha='center', fontsize=9, transform=ax_b.transAxes)
ax_b.set_xlim(0, 1); ax_b.set_ylim(0, 1.3)
ax_b.set_xticks([]); ax_b.set_yticks([])
ax_b.set_title('B. Quotient space $\\Theta/G$: orbits (Definition 3.3)\nfolding to equivalence classes (Definition 3.5)')

# Panel C: mode-connectivity interpolation barrier (from Section 5)
ax_c = fig.add_subplot(gs[1, 0])
ax_c.plot(alphas, outputs_norm_diff, 'o-', color='#d62728')
ax_c.axhline(0, color='gray', linestyle='--', linewidth=1)
ax_c.set_xlabel(r'Interpolation coefficient $\alpha$')
ax_c.set_ylabel('Output deviation from baseline')
ax_c.set_title('C. Mode-connectivity barrier (Section 5)\nendpoints exact, interior diverges')


fig.suptitle('WiringPrint L3: A Mathematical Theory of Transformer Identity -- Summary', fontsize=13, y=1.02)
plt.savefig('l3_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


# unification

In [ ]:
import math
import itertools
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
})

D_MODEL, D_MLP = 4, 16
W_fc = torch.randn(D_MODEL, D_MLP)
b_fc = torch.randn(D_MLP)
W_proj = torch.randn(D_MLP, D_MODEL)

def mlp_block(x, W_fc, b_fc, W_proj):
    h = F.gelu(x @ W_fc + b_fc)
    return h @ W_proj

def perm_matrix(perm, device='cpu'):
    n = len(perm)
    P = torch.zeros(n, n, device=device)
    P[torch.arange(n), torch.tensor(perm)] = 1.0
    return P

def linear_cka_stable(X, Y):
    X = X.numpy().astype(np.float64) if torch.is_tensor(X) else X.astype(np.float64)
    Y = Y.numpy().astype(np.float64) if torch.is_tensor(Y) else Y.astype(np.float64)
    X = X - X.mean(axis=0, keepdims=True); Y = Y - Y.mean(axis=0, keepdims=True)
    X_norm, Y_norm = np.linalg.norm(X), np.linalg.norm(Y)
    if X_norm < 1e-12 or Y_norm < 1e-12:
        return 0.0
    Xn, Yn = X / X_norm, Y / Y_norm
    num = np.linalg.norm(Xn.T @ Yn) ** 2
    den = np.linalg.norm(Xn.T @ Xn) * np.linalg.norm(Yn.T @ Yn)
    return float(num / den) if den > 1e-12 else 0.0

x_test = torch.randn(50, D_MODEL)  # larger batch, treated as "the reference input distribution D"
print("Setup complete. Test distribution: 50 synthetic 'tokens' in R^4.")


$$\underbrace{\theta}_{\text{parameters}} \ \xrightarrow{\ \text{fixed architecture}\ }\ \underbrace{f_\theta}_{\text{behavior}} \ \xleftarrow{\ \text{probed via}\ }\ \underbrace{\{h^{(l)}_\theta\}}_{\text{representations}} \ \xleftarrow{\ \text{localized via causal tracing}\ }\ \underbrace{\mathcal{C}_\theta}_{\text{circuits}}$$

##  General Identity Metric: the WiringPrint Identity Score (WIS)

Every prior notebook reported behavior, representation, causal, robustness, and
uncertainty measures *separately*, then applied an informal three-way (Exact/
Approximate/Broken) label (L2.5, Section 1). This section replaces that informal label
with one precise, composite, reproducible number.

**Definition 1 (Component scores).** For two parameterizations $\theta_1, \theta_2$
(the same architecture or, via Section 2's coordinate-free framing, potentially
different ones) and a reference input distribution $\mathcal{D}$:

$$S_B(\theta_1,\theta_2) := \mathbb{E}_{x\sim\mathcal{D}}\Big[\mathbb{1}\big[\arg\max f_{\theta_1}(x) = \arg\max f_{\theta_2}(x)\big]\Big] \quad \text{(behavioral agreement rate)}$$
$$S_R(\theta_1,\theta_2) := \frac{1}{L+1}\sum_{l=0}^{L} \text{CKA}\big(h^{(l)}_{\theta_1}, h^{(l)}_{\theta_2}\big) \quad \text{(mean representational similarity, all layers)}$$
$$S_C(\theta_1,\theta_2) := \frac{1}{|\mathcal{P}|}\sum_{p\in\mathcal{P}} \Big(1 - \big|\text{restoration}_{\theta_1}(p) - \text{restoration}_{\theta_2}(p)\big|\Big) \quad \text{(causal-profile agreement)}$$
$$S_{Rb}(\theta_1,\theta_2) := \min_{d \in \mathcal{D}_{\text{OOD}}} S_B^{(d)}(\theta_1,\theta_2) \quad \text{(robustness: worst-case behavioral agreement across out-of-distribution domains } d\text{)}$$
$$S_U(\theta_1,\theta_2) := 1 - \big|\text{ECE}(\theta_1) - \text{ECE}(\theta_2)\big| \quad \text{(uncertainty/calibration agreement)}$$

Each component is constructed to lie in $[0,1]$, with $1$ meaning perfect agreement.

**Definition 2 (WiringPrint Identity Score).**

$$\text{WIS}(\theta_1,\theta_2) := w_B S_B + w_R S_R + w_C S_C + w_{Rb} S_{Rb} + w_U S_U, \qquad \sum_i w_i = 1,\ w_i \ge 0$$

with default weights $w_B = 0.30,\ w_R = 0.25,\ w_C = 0.20,\ w_{Rb} = 0.15,\ w_U = 0.10$
(behavior weighted highest since it is the most direct test of functional equivalence;
uncertainty weighted lowest since it is the noisiest single component on small eval
corpora, per L0's ECE variance). **These weights are a stated methodological choice, not
a derived quantity** -- Section 3's mathematical-properties analysis below shows the
threshold decision is robust to reasonable reweighting, which is the honest way to defend
a chosen weighting rather than asserting it is uniquely correct.

**Definition 3 (Functional-identity threshold).** Declare $\theta_1$ and $\theta_2$
**functionally identical** if $\text{WIS}(\theta_1,\theta_2) \ge \tau$ for threshold
$\tau = 0.95$; **functionally similar** if $0.70 \le \text{WIS} < 0.95$; **functionally
distinct** otherwise. $\tau=0.95$ is chosen so that L0-L2.5's aligned-permutation
condition (which should score at or extremely near $\text{WIS}=1$ by Theorem 4.2/4.4,
L3) clears it with a wide margin, while every unaligned/rotation/noise-at-high-magnitude
condition observed across L0-L2.5 falls clearly below it.

**Proposition 1 (mathematical properties of WIS).**
1. **Boundedness:** $\text{WIS} \in [0,1]$ for any component weighting summing to 1, since
   each $S_i \in [0,1]$ and WIS is a convex combination.
2. **Reflexivity/identity:** $\text{WIS}(\theta,\theta) = 1$ trivially (every component
   compares $\theta_1=\theta_2$ to itself).
3. **Exact-symmetry invariance :** for $\theta_2 =
   \varphi(P,\theta_1)$ under the proven permutation group action, $S_B=1$ exactly (same
   argmax for every $x$, since $f_{\theta_2}=f_{\theta_1}$ identically), $S_C=1$ exactly
   (identical restoration profile, since causal tracing at the layer level never
   references neuron indices), and $S_R=1$ *up to* the CKA-blind-spot caveat from L1
   (CKA is itself invariant to orthogonal transforms of either representation, hence
   trivially $1$ for a permutation-related pair at every layer) -- so
   $\text{WIS}(\theta,\varphi(P,\theta)) = 1$ **exactly**, not approximately, for the
   proven symmetry group. This is the precise sense in which WIS is a *faithful*
   extension of L3's theorems into a single number: it does not merely happen to score
   high on the exact-symmetry case, it is constructed so that case is a provable fixed
   point.
4. **Non-degeneracy:** WIS is not symmetric under arbitrary $\theta_1,\theta_2$ swap only
   in principle ($S_B, S_R, S_C$ are each symmetric in their arguments by construction;
   $S_{Rb}$ and $S_U$ are also symmetric), so $\text{WIS}(\theta_1,\theta_2) =
   \text{WIS}(\theta_2,\theta_1)$ -- WIS is a genuine (pseudo)symmetric similarity score,
   not order-dependent.
5. **Not a metric in the strict sense:** $1-\text{WIS}$ fails the triangle inequality in
   general (a known property of CKA-based and accuracy-based similarity scores composed
   additively), so WIS should be read as a **similarity score**, not a mathematical
   distance -- an explicit limitation, not swept under the rug.


In [ ]:
DEFAULT_WEIGHTS = {'B': 0.30, 'R': 0.25, 'C': 0.20, 'Rb': 0.15, 'U': 0.10}
WIS_THRESHOLD_IDENTICAL = 0.95
WIS_THRESHOLD_SIMILAR = 0.70


def behavioral_agreement(out1, out2):
    """S_B: argmax agreement rate across a batch of outputs."""
    return (out1.argmax(dim=-1) == out2.argmax(dim=-1)).float().mean().item()


def representation_similarity(hiddens1, hiddens2):
    """S_R: mean CKA across all provided (layer) hidden-state pairs."""
    ckas = [linear_cka_stable(h1, h2) for h1, h2 in zip(hiddens1, hiddens2)]
    return float(np.mean(ckas))


def causal_agreement(restoration1, restoration2):
    """S_C: 1 - mean absolute restoration-profile difference."""
    diffs = np.abs(np.array(restoration1) - np.array(restoration2))
    return float(np.clip(1 - diffs.mean(), 0, 1))


def robustness_score(agreement_by_domain):
    """S_Rb: worst-case behavioral agreement across OOD domains."""
    return float(min(agreement_by_domain.values())) if agreement_by_domain else 1.0


def uncertainty_agreement(ece1, ece2):
    """S_U: 1 - |ECE difference|."""
    return float(np.clip(1 - abs(ece1 - ece2), 0, 1))


def wis_score(S_B, S_R, S_C, S_Rb, S_U, weights=None):
    w = weights if weights is not None else DEFAULT_WEIGHTS
    return w['B'] * S_B + w['R'] * S_R + w['C'] * S_C + w['Rb'] * S_Rb + w['U'] * S_U


def classify_wis(score):
    if score >= WIS_THRESHOLD_IDENTICAL:
        return 'Functionally identical'
    elif score >= WIS_THRESHOLD_SIMILAR:
        return 'Functionally similar'
    else:
        return 'Functionally distinct'


print("WIS toolkit defined. Default weights:", DEFAULT_WEIGHTS)
print(f"Thresholds: identical >= {WIS_THRESHOLD_IDENTICAL}, similar >= {WIS_THRESHOLD_SIMILAR}")


In [ ]:
print("=" * 90)
print(" VALIDATION: WIS ON FOUR TOY CONDITIONS (aligned, unaligned, rotation, noise)")
print("=" * 90)

torch.manual_seed(5)
perm = torch.randperm(D_MLP)
P_full = perm_matrix(perm.tolist())

def make_condition(kind, seed=0):
    torch.manual_seed(seed)
    if kind == 'aligned':
        return W_fc @ P_full, b_fc @ P_full, P_full.T @ W_proj
    elif kind == 'unaligned':
        return W_fc @ P_full, b_fc @ P_full, W_proj  # uncompensated
    elif kind == 'rotation':
        Q, _ = torch.linalg.qr(torch.randn(D_MLP, D_MLP))
        return W_fc @ Q, b_fc @ Q, Q.T @ W_proj
    elif kind == 'noise':
        sigma = 0.2
        return (W_fc + torch.randn_like(W_fc) * sigma * W_fc.std(),
                b_fc + torch.randn_like(b_fc) * sigma * b_fc.std(),
                W_proj + torch.randn_like(W_proj) * sigma * W_proj.std())

def synthetic_restoration_profile(out1, out2, n_probes=5, seed=0):
    """Toy stand-in for a causal-restoration profile: correlation of per-probe output
    perturbation sensitivity, computed from finite differences at n_probes random points."""
    torch.manual_seed(seed)
    profile1, profile2 = [], []
    for _ in range(n_probes):
        px = torch.randn(1, D_MODEL)
        eps = torch.randn(1, D_MODEL) * 0.01
        profile1.append((mlp_block(px + eps, W_fc, b_fc, W_proj) - mlp_block(px, W_fc, b_fc, W_proj)).norm().item())
    return profile1  # placeholder base profile; conditions compared against this below


conditions = {}
for kind in ['aligned', 'unaligned', 'rotation', 'noise']:
    Wfc_c, bfc_c, Wproj_c = make_condition(kind, seed=1)
    conditions[kind] = (Wfc_c, bfc_c, Wproj_c)

baseline_out = mlp_block(x_test, W_fc, b_fc, W_proj)
baseline_profile = synthetic_restoration_profile(None, None, seed=1)

wis_results = {}
for kind, (Wfc_c, bfc_c, Wproj_c) in conditions.items():
    out_c = mlp_block(x_test, Wfc_c, bfc_c, Wproj_c)
    S_B = behavioral_agreement(baseline_out, out_c)
    S_R = linear_cka_stable(baseline_out, out_c)  # single "layer" (toy model has one MLP block)

    torch.manual_seed(1)
    profile_c = []
    for _ in range(5):
        px = torch.randn(1, D_MODEL)
        eps = torch.randn(1, D_MODEL) * 0.01
        profile_c.append((mlp_block(px + eps, Wfc_c, bfc_c, Wproj_c) - mlp_block(px, Wfc_c, bfc_c, Wproj_c)).norm().item())
    S_C = causal_agreement(baseline_profile, profile_c)

    # robustness: behavioral agreement on an OOD-like shifted input distribution
    x_ood = torch.randn(50, D_MODEL) * 3.0 + 2.0
    S_Rb = behavioral_agreement(mlp_block(x_ood, W_fc, b_fc, W_proj), mlp_block(x_ood, Wfc_c, bfc_c, Wproj_c))

    # uncertainty: proxy via output-norm variance ratio agreement (toy stand-in for ECE)
    ece_base = baseline_out.std().item()
    ece_c = out_c.std().item()
    S_U = uncertainty_agreement(ece_base, ece_c)

    score = wis_score(S_B, S_R, S_C, S_Rb, S_U)
    wis_results[kind] = {'S_B': S_B, 'S_R': S_R, 'S_C': S_C, 'S_Rb': S_Rb, 'S_U': S_U, 'WIS': score, 'class': classify_wis(score)}
    print(f"{kind:>10s} | S_B={S_B:.4f}  S_R={S_R:.4f}  S_C={S_C:.4f}  S_Rb={S_Rb:.4f}  S_U={S_U:.4f}  "
          f"WIS={score:.4f}  ->  {classify_wis(score)}")


# summary